In [50]:
#imports
import sys
import os
import re
import importlib
import cv2
import torch
import torchvision
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from run_on_video.run import MomentDETRPredictor

from cv_utils import *

ModuleNotFoundError: No module named 'run_on_video'

In [12]:
# Fleiss Kappa
def load_video_annotations(video_path):
    df = pd.read_csv(video_path)
    df.columns = ['marius', 'ionas', 'angelos', 'combined_salience', 'event', 'timestamp']

    return df

def get_fleiss_kappa(video_path): 
    df = load_video_annotations(video_path)
    new_df = pd.DataFrame(columns = ['score_1', 'score_2', 'score_3'])

    for idx, row in df.iterrows():
        tmp = row["combined_salience"]
        subjectivity = tmp[2:-1].split(';')

        new_df.loc[idx] = [float(score) for score in subjectivity]

    kappa_data, categories = aggregate_raters(new_df)
    kappa = fleiss_kappa(kappa_data, method="fleiss")
    print(f"Fleiss kappa score for video: {video_path} \n - {kappa:.3f}")

annotations_path = []
annotation_directory = "annotations/"
for annot_file in os.listdir(annotation_directory):
    if 'combined' in annot_file:
        path = os.path.join(annotation_directory, annot_file)
        annotations_path.append(path)

[get_fleiss_kappa(path) for path in annotations_path]

Fleiss kappa score for video: annotations/03-TVSUM-Video_09-combined.txt 
 - 0.181
Fleiss kappa score for video: annotations/03-TVSUM-Video_10_combined.txt 
 - 0.222
Fleiss kappa score for video: annotations/03-TVSUM-Video_11-combined.txt 
 - 0.518
Fleiss kappa score for video: annotations/03-TVSUM-Video_12_combined.txt 
 - 0.252


[None, None, None, None]

# Task 3: Event Detection Using Vision-Language Models

This notebook performs VLM-based event detection for the Group 3 TVSUM videos:
- TVSUM 9
- TVSUM 10
- TVSUM 11
- TVSUM 12

This notebook completes Task 3 of the assignment:
1. It samples video frames.
2. It runs Qwen2.5-VL for event-only detection.
3. It runs Qwen2.5-VL for event + timestamp detection.
4. It parses VLM outputs into structured CSV files.
5. It parses manual annotations.
6. It compares VLM events against manual annotations using sentence embedding similarity.
7. It evaluates prompt strategies: zero-shot, few-shot, and reasoning-guided prompting.
8. It performs failure analysis by listing missed manual events and grouping them by subjectivity.

Expected folder structure:

```text
Comp Vision/
    vlm_event_detection_group3.ipynb
    videos/
        video_9.mp4
        video_10.mp4
        video_11.mp4
        video_12.mp4
```

In [18]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA GeForce RTX 3070


In [19]:
GROUP_ID = "Group3"
DATASET_NAME = "TVSUM"

VIDEO_DIR = Path("videos")

OUTPUT_DIR = output_dir("outputs_vlm_event_detection")

VIDEOS = {
    "TVSUM_9": VIDEO_DIR / "video_9.mp4",
    "TVSUM_10": VIDEO_DIR / "video_10.mp4",
    "TVSUM_11": VIDEO_DIR / "video_11.mp4",
    "TVSUM_12": VIDEO_DIR / "video_12.mp4",
}

PLOT=False

FRAME_EVERY_N_SECONDS = 2

MAX_FRAMES_FOR_VLM = 12

In [20]:
# Checking Video Paths
for video_name, video_path in VIDEOS.items():
    print(video_name, "->", video_path)

    if video_path.exists():
        print("  Found")
    else:
        print("  NOT FOUND")

TVSUM_9 -> videos\video_9.mp4
  Found
TVSUM_10 -> videos\video_10.mp4
  Found
TVSUM_11 -> videos\video_11.mp4
  Found
TVSUM_12 -> videos\video_12.mp4
  Found


# Frame Sampling

Instead of passing the full video directly to the VLM, we sample frames at regular intervals.

The frames are sampled every 2 seconds and then uniformly reduced to a maximum of 12 frames. This keeps the input small enough for the model while still covering the full video.

In [21]:
def sample_frames_from_video(video_path, output_frame_dir, every_n_seconds=2, max_frames=None):
    """
    Sample frames from a video every N seconds.

    Parameters
    ----------
    video_path:
        Path to the input video.

    output_frame_dir:
        Folder where sampled frames will be saved.

    every_n_seconds:
        Sampling interval in seconds.

    max_frames:
        Maximum number of frames to keep.
        If the video produces more frames, we uniformly select max_frames.

    Returns
    -------
    sampled_frames:
        A list of dictionaries containing timestamp, seconds, and frame path.
    """

    output_frame_dir = output_dir(output_frame_dir)
    video_path = load_vid_pth(video_path)
    
    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    if fps == 0:
        raise ValueError(f"Could not read FPS from video: {video_path}")

    duration_seconds = frame_count / fps

    sampled_frames = []

    timestamps = np.arange(0, duration_seconds, every_n_seconds)

    # If too many frames, select max_frames uniformly across the whole video
    if max_frames is not None and len(timestamps) > max_frames:
        indices = np.linspace(0, len(timestamps) - 1, max_frames).astype(int)
        timestamps = timestamps[indices]

    for t in timestamps:
        frame_index = int(t * fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)

        success, frame = cap.read()

        if not success:
            continue

        # OpenCV reads in BGR, but PIL/matplotlib use RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        timestamp = seconds_to_mmss(t)
        frame_filename = f"frame_{timestamp.replace(':', '-')}.jpg"
        frame_path = output_frame_dir / frame_filename

        Image.fromarray(frame_rgb).save(frame_path)

        sampled_frames.append({
            "timestamp": timestamp,
            "seconds": int(t),
            "frame_path": str(frame_path)
        })

    cap.release()

    return sampled_frames

In [43]:
# Test frame sampling on TVSUM 9

video_name = "TVSUM_12"
video_path = VIDEOS[video_name]

frame_dir = OUTPUT_DIR / video_name / "sampled_frames"

sampled_frames = sample_frames_from_video(
    video_path=video_path,
    output_frame_dir=frame_dir,
    every_n_seconds=FRAME_EVERY_N_SECONDS,
    max_frames=MAX_FRAMES_FOR_VLM
)

sampled_frames_df = pd.DataFrame(sampled_frames)
if PLOT:
    display(sampled_frames_df)

In [44]:
# Visualizing Sampled Frames
def display_sampled_frames(sampled_frames, cols=4):
    """
    Display sampled frames with timestamps.
    """

    n = len(sampled_frames)

    if n == 0:
        print("No frames to display.")
        return

    rows = int(np.ceil(n / cols))

    plt.figure(figsize=(4 * cols, 3 * rows))

    for i, item in enumerate(sampled_frames):
        image = Image.open(item["frame_path"])

        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(item["timestamp"])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

if PLOT:
    display_sampled_frames(sampled_frames)

In [ ]:
EVENT_ONLY_PROMPT = """
You are given sampled frames from a video in chronological order.

Task:
Retrieve all of the salient events of the video.

Rules:
- List only important events needed to understand the video.
- Keep the events in chronological order.
- Use short and clear descriptions.
- Do not describe every frame.
- Do not include timestamps.
- Do not include unimportant visual details.
- Use this exact output format:

Salient event 1: <event description>
Salient event 2: <event description>
Salient event 3: <event description>
"""


EVENT_TIMESTAMP_PROMPT = """
You are given sampled frames from a video in chronological order.
Each frame is labeled with its timestamp.

Task:
Retrieve all of the salient events of the video with temporal localization.

Rules:
- List only important events needed to understand the video.
- Keep events in chronological order.
- Use short and clear descriptions.
- Use precise but realistic timestamps.
- Use MM:SS - MM:SS format.
- Do not describe every frame.
- Do not include unimportant visual details.
- Use this exact output format:

Salient event 1: <event description>, <MM:SS - MM:SS>
Salient event 2: <event description>, <MM:SS - MM:SS>
Salient event 3: <event description>, <MM:SS - MM:SS>
"""

print(EVENT_ONLY_PROMPT)
print(EVENT_TIMESTAMP_PROMPT)


You are given sampled frames from a video in chronological order.

Task:
Retrieve all of the salient events of the video.

Rules:
- List only important events needed to understand the video.
- Keep the events in chronological order.
- Use short and clear descriptions.
- Do not describe every frame.
- Do not include timestamps.
- Do not include unimportant visual details.
- Use this exact output format:

Salient event 1: <event description>
Salient event 2: <event description>
Salient event 3: <event description>


You are given sampled frames from a video in chronological order.
Each frame is labeled with its timestamp.

Task:
Retrieve all of the salient events of the video with temporal localization.

Rules:
- List only important events needed to understand the video.
- Keep events in chronological order.
- Use short and clear descriptions.
- Use precise but realistic timestamps.
- Use start - end format.
- use only minutes and seconds in the timestamp
- Do not describe every frame.


In [46]:
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

if device == "cpu":
    model = model.to(device)

Using device: cuda


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [47]:
def build_vlm_message(sampled_frames, prompt):
    """
    Build a Qwen2.5-VL message containing sampled frames and a text prompt.
    """

    content = []

    for item in sampled_frames:
        content.append({
            "type": "text",
            "text": f"Frame from {item['timestamp']}:"
        })

        content.append({
            "type": "image",
            "image": item["frame_path"]
        })

    content.append({
        "type": "text",
        "text": prompt
    })

    messages = [
        {
            "role": "user",
            "content": content
        }
    ]

    return messages

In [48]:
def run_qwen_vlm(sampled_frames, prompt, max_new_tokens=512):
    """
    Run Qwen2.5-VL on sampled frames using the given prompt.
    """

    messages = build_vlm_message(sampled_frames, prompt)

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return output_text

In [41]:
# Run event-only prompt on TVSUM 9 for testing 

event_only_output = run_qwen_vlm(
    sampled_frames=sampled_frames,
    prompt=EVENT_ONLY_PROMPT,
    max_new_tokens=512
)

print(event_only_output)

Salient event 1: A woman is seen grooming a dog in a pet salon.
Salient event 2: The woman is using various tools and products to groom the dog.
Salient event 3: The woman is also seen interacting with other dogs in the salon.


In [49]:
# Run event + timestamp prompt on TVSUM 9 for testing 

event_timestamp_output = run_qwen_vlm(
    sampled_frames=sampled_frames,
    prompt=EVENT_TIMESTAMP_PROMPT,
    max_new_tokens=512
)

print(event_timestamp_output)

Salient event 1: Person washing hands, 00:00 - 00:05
Salient event 2: Woman talking to camera in kitchen, 00:10 - 00:20
Salient event 3: Woman showing products on counter, 00:20 - 00:30
Salient event 4: Woman pouring liquid into sink, 00:30 - 00:40
Salient event 5: Woman talking to camera again, 00:40 - 00:50
Salient event 6: Woman showing dog toys, 00:50 - 01:00
Salient event 7: Woman pouring liquid into sink again, 01:00 - 01:10
Salient event 8: Woman talking to camera, 01:10 - 01:20
Salient event 9: Woman showing dog toys again, 01:20 - 01:30
Salient event 10: Woman pouring liquid into sink once more, 01:30 - 01:40
Salient event 11: Woman talking to camera, 01:40 - 01:50
Salient event 12: Woman shows dog toys again, 01:50 - 02:00
Salient event 13: Woman pours liquid into sink for third time, 02:00 - 02:10
Salient event 14: Woman talks to camera, 02:10 - 02:20
Salient event 15: Woman shows dog toys again, 02:20 - 02:30
Salient event 16: Woman pours liquid into sink for fourth time, 0

In [ ]:
# Saving Raw VLM Outputs
def save_text(text, path):
    """
    Save text to a file.
    """

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


video_output_dir = output_dir(OUTPUT_DIR / video_name)

save_text(event_only_output, video_output_dir / "vlm_event_only_raw.txt")
save_text(event_timestamp_output, video_output_dir / "vlm_event_timestamp_raw.txt")

In [ ]:
def parse_event_only(vlm_text):
    """
    Parse event-only VLM output.

    Expected format:
    Salient event 1: description
    Salient event 2: description

    Returns:
    DataFrame with columns:
    - event_id
    - event_description
    """

    pattern = r"Salient event\s*(\d+)\s*:\s*(.+)"
    rows = []

    for match in re.finditer(pattern, vlm_text, flags=re.IGNORECASE):
        event_id = int(match.group(1))
        description = match.group(2).strip()

        # Remove accidental timestamp if model included one
        description = re.sub(
            r",?\s*\d{2}:\d{2}\s*-\s*\d{2}:\d{2}",
            "",
            description
        ).strip()

        rows.append({
            "event_id": event_id,
            "event_description": description
        })

    return pd.DataFrame(rows)

In [ ]:

def parse_event_with_timestamps(vlm_text):
    """
    Flexible parser for timestamped VLM output.

    Handles:
    Salient event 1: description, 00:00 - 00:20
    Salient event 1: description, MM:00:00 - MM:00:20
    Salient event 1: description, 00:00–00:20
    """

    rows = []

    # Normalize different dash symbols
    vlm_text = normalise_dash(vlm_text)

    pattern = (
        r"Salient event\s*(\d+)\s*:\s*"
        r"(.+),\s*"
        r"(?:MM:)?(\d{1,2}:\d{2})\s*-\s*"
        r"(?:MM:)?(\d{1,2}:\d{2})")

    for match in re.finditer(pattern, vlm_text, flags=re.IGNORECASE):
        event_id = int(match.group(1))
        description = match.group(2).strip()
        start = clean_timestamp(match.group(3))
        end = clean_timestamp(match.group(4))

        rows.append({
            "event_id": event_id,
            "event_description": description,
            "start": start,
            "end": end,
            "start_seconds": mmss_to_seconds_flexible(start),
            "end_seconds": mmss_to_seconds_flexible(end)
        })

    return pd.DataFrame(rows)

In [ ]:
parsed_event_only_df = parse_event_only(event_only_output)
parsed_timestamp_df = parse_event_with_timestamps(event_timestamp_output)

print("Event-only parsed output:")
display(parsed_event_only_df)

print("Event + timestamp parsed output:")
display(parsed_timestamp_df)

Event-only parsed output:


,event_id,event_description
0,1,A man wearing glasses is standing next to a wh...
1,2,The scene transitions to a computer-generated ...
2,3,The video then shows a close-up of the interio...
3,4,The video returns to the man standing next to ...
4,5,The video then displays an animation of the ve...
5,6,The video shifts to a woman sitting inside the...
6,7,The video concludes with a news segment featur...


Event + timestamp parsed output:


,event_id,event_description,start,end,start_seconds,end_seconds
0,1,A man wearing glasses and a black jacket is ta...,00:00,00:20,0,20
1,2,The scene transitions to a futuristic car driv...,00:20,00:42,20,42
2,3,"The camera focuses on the interior of the car,...",00:42,01:02,42,62
3,4,The video then shows a close-up of the car's w...,01:02,01:24,62,84
4,5,The scene returns to the man talking,01:24,01:44,84,104
5,6,The video then shows a diagram of the car's in...,01:44,02:06,104,126
6,7,The video cuts back to the man talking,02:06,02:26,126,146
7,8,The video then shows a diagram of the car's ba...,02:26,02:48,146,168
8,9,The video cuts back to the man talking,02:48,03:08,168,188
9,10,The video then shows a person sitting inside t...,03:08,03:30,188,210


In [ ]:
raw_path = OUTPUT_DIR / "TVSUM_9" / "vlm_event_timestamp_raw.txt"

with open(raw_path, "r", encoding="utf-8") as f:
    raw_timestamp_9 = f.read()

print(raw_timestamp_9)

Salient event 1: Man sitting next to a car, talking, MM:00:00 - MM:00:20
Salient event 2: Car driving on a road, MM:00:20 - MM:00:42
Salient event 3: Close-up of car pedals, MM:00:42 - MM:01:02
Salient event 4: Man sitting next to a car again, MM:01:02 - MM:01:24
Salient event 5: Car driving on a road again, MM:01:24 - MM:01:44
Salient event 6: Close-up of car pedals again, MM:01:44 - MM:02:06
Salient event 7: Man sitting next to a car again, MM:02:06 - MM:02:26
Salient event 8: Car driving on a road again, MM:02:26 - MM:02:48
Salient event 9: Close-up of car components, MM:02:48 - MM:03:08
Salient event 10: Woman driving a futuristic car, MM:03:08 - MM:03:30
Salient event 11: Intel Free Press Tech News banner, MM:03:30 - MM:03:52


In [ ]:
parsed_timestamp_9 = parse_event_with_timestamps(raw_timestamp_9)
display(parsed_timestamp_9)

,event_id,event_description,start,end,start_seconds,end_seconds
0,1,"Man sitting next to a car, talking",00:00,00:20,0,20
1,2,Car driving on a road,00:20,00:42,20,42
2,3,Close-up of car pedals,00:42,01:02,42,62
3,4,Man sitting next to a car again,01:02,01:24,62,84
4,5,Car driving on a road again,01:24,01:44,84,104
5,6,Close-up of car pedals again,01:44,02:06,104,126
6,7,Man sitting next to a car again,02:06,02:26,126,146
7,8,Car driving on a road again,02:26,02:48,146,168
8,9,Close-up of car components,02:48,03:08,168,188
9,10,Woman driving a futuristic car,03:08,03:30,188,210


In [ ]:
parsed_event_only_df.to_csv(
    video_output_dir / "parsed_event_only.csv",
    index=False
)

parsed_timestamp_df.to_csv(
    video_output_dir / "parsed_event_timestamp.csv",
    index=False
)

In [ ]:
def run_and_parse_pipeline(video_name, video_path, output_dir, prompt_name, prompt_text, parser_func):
    """
    Generic pipeline to sample frames, run the VLM, save raw text, and parse the output.
    """
    video_output_dir = output_dir / video_name
    video_output_dir.mkdir(parents=True, exist_ok=True)
    frame_dir = video_output_dir / "sampled_frames"

    sampled_frames = sample_frames_from_video(
        video_path=video_path,
        output_frame_dir=frame_dir,
        every_n_seconds=FRAME_EVERY_N_SECONDS,
        max_frames=MAX_FRAMES_FOR_VLM
    )
    
    sampled_frames_df = pd.DataFrame(sampled_frames)
    sampled_frames_df.to_csv(video_output_dir / "sampled_frames_metadata.csv", index=False)

    vlm_output = run_qwen_vlm(
        sampled_frames=sampled_frames,
        prompt=prompt_text,
        max_new_tokens=512
    )
    
    save_text(vlm_output, video_output_dir / f"vlm_raw_{prompt_name}.txt")

    parsed_df = parser_func(vlm_output)
    
    if parsed_df.empty:
        print(f"\nWARNING: Parser found no events for {prompt_name}.")
        print(vlm_output)

    parsed_df.insert(0, "video_name", video_name)
    parsed_df.insert(1, "prompt_name", prompt_name)
    parsed_df.to_csv(video_output_dir / f"parsed_{prompt_name}.csv", index=False)

    return sampled_frames_df, vlm_output, parsed_df

In [ ]:
def process_video_until_parsing(video_name, video_path):
    """
    Runs event-only prompt
    Runs event+timestamp prompt
    """
    print(f"\n==============================")
    print(f"Processing {video_name}")
    print(f"==============================")

    # Event-Only
    print("Event-only prompt")
    sampled_frames_df, event_only_output, parsed_event_only_df = run_and_parse_pipeline(
        video_name=video_name,
        video_path=video_path,
        output_dir=OUTPUT_DIR,
        prompt_name="event_only",
        prompt_text=EVENT_ONLY_PROMPT,
        parser_func=parse_event_only
    )
    
    print("\nEvent-only parsed output:")
    display(parsed_event_only_df)

    # Event + Timestamp Pipeline
    print("Event + timestamp prompt")
    _, event_timestamp_output, parsed_timestamp_df = run_and_parse_pipeline(
        video_name=video_name,
        video_path=video_path,
        output_dir=OUTPUT_DIR,
        prompt_name="event_timestamp",
        prompt_text=EVENT_TIMESTAMP_PROMPT,
        parser_func=parse_event_with_timestamps
    )
    
    print("\nEvent + timestamp parsed output:")
    display(parsed_timestamp_df)

    return {
        "video_name": video_name,
        "sampled_frames": sampled_frames_df,
        "event_only_output": event_only_output,
        "event_timestamp_output": event_timestamp_output,
        "parsed_event_only": parsed_event_only_df,
        "parsed_timestamp": parsed_timestamp_df
    }

In [ ]:
all_parsed_results = {}

for video_name, video_path in VIDEOS.items():
    result = process_video_until_parsing(video_name, video_path)
    all_parsed_results[video_name] = result


Processing TVSUM_9
Sampled 12 frames.
Running event-only prompt...
Running event + timestamp prompt...

Event-only parsed output:


,event_id,event_description
0,1,A man wearing glasses is sitting next to a whi...
1,2,The scene changes to a futuristic-looking vehi...
2,3,"The interior of the vehicle is shown, highligh..."
3,4,The man is then seen driving the vehicle throu...
4,5,The video concludes with a news segment featur...



Event + timestamp parsed output:


,event_id,event_description,start,end,start_seconds,end_seconds
0,1,"Man sitting next to a car, talking",00:00,00:20,0,20
1,2,Car driving on a road,00:20,00:42,20,42
2,3,Close-up of car pedals,00:42,01:02,42,62
3,4,Man sitting next to a car again,01:02,01:24,62,84
4,5,Car driving on a road again,01:24,01:44,84,104
5,6,Close-up of car pedals again,01:44,02:06,104,126
6,7,Man sitting next to a car again,02:06,02:26,126,146
7,8,Car driving on a road again,02:26,02:48,146,168
8,9,Close-up of car components,02:48,03:08,168,188
9,10,Woman driving a futuristic car,03:08,03:30,188,210



Processing TVSUM_10
Sampled 12 frames.
Running event-only prompt...
Running event + timestamp prompt...

Event-only parsed output:


...

,event_id,event_description,start,end,start_seconds,end_seconds
0,1,Woman entering a pet grooming salon,00:00,00:14,0,14
1,2,Woman preparing a dog for grooming,00:14,00:28,14,28
2,3,Woman grooming a golden retriever,00:28,00:56,28,56
3,4,Woman grooming a small grey dog,00:56,01:10,56,70
4,5,Woman talking about pet food options,01:10,01:24,70,84
5,6,Woman showing different types of pet food,01:24,02:06,84,126
6,7,Woman closing the dog's cage,02:06,02:36,126,156
7,8,Woman exiting the pet grooming salon,02:36,02:40,156,160



Processing TVSUM_12
Sampled 12 frames.
Running event-only prompt...
Running event + timestamp prompt...

Event-only parsed output:


,event_id,event_description
0,1,A person is washing a small white dog in a sink.
1,2,The person applies a thickening agent to the d...
2,3,The person dries the dog with a towel after ap...



Event + timestamp parsed output:


,event_id,event_description,start,end,start_seconds,end_seconds
0,1,Person washing dog's fur,00:00,00:09,0,9
1,2,Woman talking to camera,00:40,00:45,40,45
2,3,Woman showing products,01:00,01:10,60,70
3,4,Woman giving dog bath,01:20,02:00,80,120
4,5,Woman talking to camera,02:00,02:05,120,125
5,6,Woman giving dog bath,02:10,02:40,130,160
6,7,Woman talking to camera,02:40,02:45,160,165
7,8,Woman giving dog bath,02:50,03:20,170,200
8,9,Woman talking to camera,03:20,03:25,200,205
9,10,Woman giving dog bath,03:30,04:00,210,240


In [ ]:
# Combine outputs into one CSV

all_event_only_dfs = []

for video_name, result in all_parsed_results.items():
    df = result["parsed_event_only"].copy()
    df.insert(0, "video_name", video_name)
    all_event_only_dfs.append(df)

all_event_only_df = pd.concat(all_event_only_dfs, ignore_index=True)
if PLOT:
    display(all_event_only_df)

all_event_only_df.to_csv(
    OUTPUT_DIR / "all_videos_parsed_event_only.csv",
    index=False
)

,video_name,event_id,event_description
0,TVSUM_9,1,A man wearing glasses is sitting next to a whi...
1,TVSUM_9,2,The scene changes to a futuristic-looking vehi...
2,TVSUM_9,3,"The interior of the vehicle is shown, highligh..."
3,TVSUM_9,4,The man is then seen driving the vehicle throu...
4,TVSUM_9,5,The video concludes with a news segment featur...
5,TVSUM_10,1,A news anchor introduces a segment about elect...
6,TVSUM_10,2,The anchor discusses the establishment of a ch...
7,TVSUM_10,3,"An interview with Yoo Dong-kun, a representati..."
8,TVSUM_10,4,The video shows a graph displaying data relate...
9,TVSUM_10,5,A business fair aimed at facilitating exchange...


In [ ]:
# Combine all timestamp outputs into one CSV

all_timestamp_dfs = []

for video_name, result in all_parsed_results.items():
    df = result["parsed_timestamp"].copy()
    df.insert(0, "video_name", video_name)
    all_timestamp_dfs.append(df)

all_timestamp_df = pd.concat(all_timestamp_dfs, ignore_index=True)
if PLOT:
    display(all_timestamp_df)

all_timestamp_df.to_csv(
    OUTPUT_DIR / "all_videos_parsed_event_timestamp.csv",
    index=False
)

,video_name,event_id,event_description,start,end,start_seconds,end_seconds
0,TVSUM_9,1,"Man sitting next to a car, talking",00:00,00:20,0,20
1,TVSUM_9,2,Car driving on a road,00:20,00:42,20,42
2,TVSUM_9,3,Close-up of car pedals,00:42,01:02,42,62
3,TVSUM_9,4,Man sitting next to a car again,01:02,01:24,62,84
4,TVSUM_9,5,Car driving on a road again,01:24,01:44,84,104
5,TVSUM_9,6,Close-up of car pedals again,01:44,02:06,104,126
6,TVSUM_9,7,Man sitting next to a car again,02:06,02:26,126,146
7,TVSUM_9,8,Car driving on a road again,02:26,02:48,146,168
8,TVSUM_9,9,Close-up of car components,02:48,03:08,168,188
9,TVSUM_9,10,Woman driving a futuristic car,03:08,03:30,188,210


In [ ]:
raw_path = OUTPUT_DIR / "TVSUM_12" / "vlm_event_timestamp_raw.txt"

with open(raw_path, "r", encoding="utf-8") as f:
    raw_timestamp_12 = f.read()

print(raw_timestamp_12)

Salient event 1: Person washing dog's fur, 00:00 - 00:09
Salient event 2: Woman talking to camera, 00:40 - 00:45
Salient event 3: Woman showing products, 01:00 - 01:10
Salient event 4: Woman giving dog bath, 01:20 - 02:00
Salient event 5: Woman talking to camera, 02:00 - 02:05
Salient event 6: Woman giving dog bath, 02:10 - 02:40
Salient event 7: Woman talking to camera, 02:40 - 02:45
Salient event 8: Woman giving dog bath, 02:50 - 03:20
Salient event 9: Woman talking to camera, 03:20 - 03:25
Salient event 10: Woman giving dog bath, 03:30 - 04:00
Salient event 11: Woman talking to camera, 04:00 - 04:05
Salient event 12: Woman giving dog bath, 04:10 - 04:40
Salient event 13: Woman talking to camera, 04:40 - 04:45
Salient event 14: Woman giving dog bath, 04:50 - 05:20
Salient event 15: Woman talking to camera, 05:20 - 05:25
Salient event 16: Woman giving dog bath, 05:30 - 06:00
Salient event 17: Woman talking to camera, 06:00 - 06:05
Salient event 18: Woman giving dog bath, 06:10 - 06:40

In [ ]:
# manual annotation parsing

ANNOTATION_PATH = Path("manual_annotations.txt")
if not ANNOTATION_PATH.exists():
    raise FileNotFoundError(f"Video not found: {ANNOTATION_PATH}")
    

def parse_manual_annotations(annotation_path):
    """
    Parse manual annotations from a txt file.

    Expected line format:
    event description, start_time - end_time, subjectivity

    Example:
    Man gets into vehicle, 00:00 - 00:06, 2

    Returns:
    DataFrame with:
    - video_name
    - event_id
    - event_description
    - start
    - end
    - start_seconds
    - end_seconds
    - subjectivity
    """

    annotation_path = Path(annotation_path)
    if not annotation_path.exists():
        raise FileNotFoundError(f"Video not found: {annotation_path}")

    with open(annotation_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    rows = []
    current_video = None
    event_counter = 0

    for line in lines:
        line = line.strip()

        if not line:
            continue

        if line.lower().startswith("event"):
            continue

        video_match = re.match(r"[-#]\s*Video\s*(\d+)", line, flags=re.IGNORECASE)

        if video_match:
            video_number = video_match.group(1)
            current_video = f"TVSUM_{video_number}"
            event_counter = 0
            continue

        if current_video is None:
            continue

        # Normalize dash symbols
        line = normalise_dash(line)

        # Split from the right:
        # description, 00:00 - 00:06, 2
        parts = [p.strip() for p in line.rsplit(",", maxsplit=2)]

        if len(parts) != 3:
            print("Could not parse annotation line:", line)
            continue

        description, timestamp_text, subjectivity = parts

        timestamp_match = re.match(
            r"(\d{1,2}:\d{2})\s*-\s*(\d{1,2}:\d{2})",
            timestamp_text
        )

        if not timestamp_match:
            print("Could not parse timestamp:", line)
            continue

        start = timestamp_match.group(1)
        end = timestamp_match.group(2)

        try:
            subjectivity = int(subjectivity)
        except Exception:
            print("Could not parse subjectivity:", line)
            subjectivity = None

        event_counter += 1

        rows.append({
            "video_name": current_video,
            "event_id": event_counter,
            "event_description": description,
            "start": start,
            "end": end,
            "start_seconds": mmss_to_seconds_flexible(start),
            "end_seconds": mmss_to_seconds_flexible(end),
            "subjectivity": subjectivity
        })

    return pd.DataFrame(rows)


manual_annotations_df = parse_manual_annotations(ANNOTATION_PATH)

if PLOT:
    print("Manual annotations:")
    display(manual_annotations_df)

manual_annotations_df.to_csv(
    OUTPUT_DIR / "manual_annotations.csv",
    index=False
)

Manual annotations:


,video_name,event_id,event_description,start,end,start_seconds,end_seconds,subjectivity
0,TVSUM_9,1,Man gets into enclosed two-wheeled vehicle,00:00,00:06,0,6,2
1,TVSUM_9,2,Man speaking to camera,00:14,00:20,14,20,1
2,TVSUM_9,3,Man gets out of enclosed two-wheeled vehicle,00:26,00:32,26,32,2
3,TVSUM_9,4,Man speaking to camera,00:33,00:42,33,42,-1
4,TVSUM_9,5,Functionality of the vehicle is explained,00:43,00:57,43,57,2
5,TVSUM_9,6,Vehicle simulation shown,00:58,01:02,58,62,2
6,TVSUM_9,7,Man speaking to camera,01:02,01:06,62,66,-1
7,TVSUM_9,8,Simulation of the inner workings of the two-wh...,01:09,01:25,69,85,2
8,TVSUM_9,9,Pedal controls shown,01:25,01:30,85,90,1
9,TVSUM_9,10,Smart display shows vehicle data,01:59,02:03,119,123,1


In [ ]:
# EMBEDDING SIMILARITY EVALUATION

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
def compute_event_similarity_matches(
    manual_df,
    vlm_df,
    similarity_threshold=0.45
):
    """
    Compare manual annotated events with VLM-generated events.

    For every manual event, find the most semantically similar VLM event
    from the same video using sentence embedding cosine similarity.

    Returns:
    DataFrame with one row per manual annotation.
    """

    rows = []

    for video_name in manual_df["video_name"].unique():
        manual_video_df = manual_df[manual_df["video_name"] == video_name].copy()
        vlm_video_df = vlm_df[vlm_df["video_name"] == video_name].copy()

        if vlm_video_df.empty:
            for _, manual_row in manual_video_df.iterrows():
                rows.append({
                    "video_name": video_name,
                    "manual_event_id": manual_row["event_id"],
                    "manual_event": manual_row["event_description"],
                    "manual_start": manual_row["start"],
                    "manual_end": manual_row["end"],
                    "subjectivity": manual_row["subjectivity"],
                    "best_vlm_event_id": None,
                    "best_vlm_event": None,
                    "similarity": 0.0,
                    "matched": False
                })
            continue

        manual_events = manual_video_df["event_description"].tolist()
        vlm_events = vlm_video_df["event_description"].tolist()

        manual_embeddings = embedding_model.encode(manual_events)
        vlm_embeddings = embedding_model.encode(vlm_events)

        similarity_matrix = cosine_similarity(manual_embeddings, vlm_embeddings)

        for i, (_, manual_row) in enumerate(manual_video_df.iterrows()):
            best_idx = int(np.argmax(similarity_matrix[i]))
            best_score = float(similarity_matrix[i][best_idx])
            best_vlm_row = vlm_video_df.iloc[best_idx]

            rows.append({
                "video_name": video_name,
                "manual_event_id": manual_row["event_id"],
                "manual_event": manual_row["event_description"],
                "manual_start": manual_row["start"],
                "manual_end": manual_row["end"],
                "subjectivity": manual_row["subjectivity"],
                "best_vlm_event_id": best_vlm_row["event_id"],
                "best_vlm_event": best_vlm_row["event_description"],
                "similarity": best_score,
                "matched": best_score >= similarity_threshold
            })

    return pd.DataFrame(rows)

In [ ]:
similarity_threshold = 0.45

event_similarity_df = compute_event_similarity_matches(
    manual_df=manual_annotations_df,
    vlm_df=all_timestamp_df,
    similarity_threshold=similarity_threshold
)
if PLOT:
    display(event_similarity_df)

event_similarity_df.to_csv(
    OUTPUT_DIR / "event_similarity_evaluation.csv",
    index=False
)

,video_name,manual_event_id,manual_event,manual_start,manual_end,subjectivity,best_vlm_event_id,best_vlm_event,similarity,matched
0,TVSUM_9,1,Man gets into enclosed two-wheeled vehicle,00:00,00:06,2,4,Man sitting next to a car again,0.409412,False
1,TVSUM_9,2,Man speaking to camera,00:14,00:20,1,1,"Man sitting next to a car, talking",0.481796,True
2,TVSUM_9,3,Man gets out of enclosed two-wheeled vehicle,00:26,00:32,2,4,Man sitting next to a car again,0.414626,False
3,TVSUM_9,4,Man speaking to camera,00:33,00:42,-1,1,"Man sitting next to a car, talking",0.481796,True
4,TVSUM_9,5,Functionality of the vehicle is explained,00:43,00:57,2,9,Close-up of car components,0.437063,False
5,TVSUM_9,6,Vehicle simulation shown,00:58,01:02,2,2,Car driving on a road,0.399613,False
6,TVSUM_9,7,Man speaking to camera,01:02,01:06,-1,1,"Man sitting next to a car, talking",0.481796,True
7,TVSUM_9,8,Simulation of the inner workings of the two-wh...,01:09,01:25,2,9,Close-up of car components,0.267351,False
8,TVSUM_9,9,Pedal controls shown,01:25,01:30,1,6,Close-up of car pedals again,0.553858,True
9,TVSUM_9,10,Smart display shows vehicle data,01:59,02:03,1,4,Man sitting next to a car again,0.277188,False


In [ ]:
# Evaluation summary

summary_rows = []

for video_name, group_df in event_similarity_df.groupby("video_name"):
    total_manual_events = len(group_df)
    matched_events = int(group_df["matched"].sum())
    missed_events = total_manual_events - matched_events

    match_rate = matched_events / total_manual_events if total_manual_events > 0 else 0

    summary_rows.append({
        "video_name": video_name,
        "manual_events": total_manual_events,
        "matched_events": matched_events,
        "missed_events": missed_events,
        "match_rate": match_rate
    })

evaluation_summary_df = pd.DataFrame(summary_rows)
if PLOT:
    display(evaluation_summary_df)

evaluation_summary_df.to_csv(
    OUTPUT_DIR / "evaluation_summary.csv",
    index=False
)

,video_name,manual_events,matched_events,missed_events,match_rate
0,TVSUM_10,6,1,5,0.166667
1,TVSUM_11,8,7,1,0.875000
2,TVSUM_12,7,2,5,0.285714
3,TVSUM_9,16,4,12,0.250000


In [ ]:
# Failure analysis
missed_events_df = event_similarity_df[
    event_similarity_df["matched"] == False
].copy()

missed_events_df = missed_events_df.sort_values(
    by=["video_name", "subjectivity", "similarity"],
    ascending=[True, False, True]
)
if PLOT:
    display(missed_events_df)

missed_events_df.to_csv(
    OUTPUT_DIR / "missed_events_failure_analysis.csv",
    index=False
)

,video_name,manual_event_id,manual_event,manual_start,manual_end,subjectivity,best_vlm_event_id,best_vlm_event,similarity,matched
21,TVSUM_10,6,Man talking with cars in the background,01:28,01:42,2,5,Overview of the electric vehicle industry,0.223952,False
18,TVSUM_10,3,Cable being plugged into a car,00:32,00:33,2,3,Demonstration of the charging process for elec...,0.369192,False
16,TVSUM_10,1,Woman talking on a set,00:05,00:25,1,9,Attendees taking notes during a presentation o...,0.294872,False
19,TVSUM_10,4,Man talking,00:46,01:03,0,9,Attendees taking notes during a presentation o...,0.262039,False
17,TVSUM_10,2,People looking at a car,00:26,00:28,0,3,Demonstration of the charging process for elec...,0.273575,False
25,TVSUM_11,4,A dog getting dried,01:30,01:32,2,2,Woman preparing a dog for grooming,0.341485,False
36,TVSUM_12,7,A pair of hair clippers being shown,06:34,06:40,2,3,Woman showing products,0.285827,False
35,TVSUM_12,6,Hair cutting materials being shown with a pile...,05:41,05:47,2,1,Person washing dog's fur,0.290694,False
34,TVSUM_12,5,An electric razor being shown next to a dog,04:31,04:34,2,1,Person washing dog's fur,0.400482,False
33,TVSUM_12,4,A dog getting pet on a kitchen countertop with...,04:25,04:30,2,1,Person washing dog's fur,0.426629,False


In [ ]:
missed_by_subjectivity_df = (
    missed_events_df
    .groupby(["video_name", "subjectivity"])
    .size()
    .reset_index(name="missed_count")
)
if PLOT:
    display(missed_by_subjectivity_df)

missed_by_subjectivity_df.to_csv(
    OUTPUT_DIR / "missed_events_by_subjectivity.csv",
    index=False
)

,video_name,subjectivity,missed_count
0,TVSUM_10,0,2
1,TVSUM_10,1,1
2,TVSUM_10,2,2
3,TVSUM_11,2,1
4,TVSUM_12,0,1
5,TVSUM_12,2,4
6,TVSUM_9,1,5
7,TVSUM_9,2,7


# Additional Prompting Strategies

To compare prompting strategies, we test two additional prompts:

1. **Few-shot prompt**: gives the model examples of the desired output style.
2. **Reasoning-guided prompt**: asks the model to internally check for distinct, important, and non-redundant events before producing the final output.

These are compared against the original zero-shot timestamp prompt.

In [ ]:
FEW_SHOT_PROMPT = """
You are given sampled frames from a video in chronological order.
Each frame is labeled with its timestamp.

Task:
Retrieve the salient events of the video with temporal localization.

Here are examples of the required output style:

Salient event 1: A person enters a vehicle, 00:00 - 00:08
Salient event 2: The vehicle interior is shown, 00:09 - 00:20
Salient event 3: The vehicle drives on the road, 00:21 - 00:35

Now analyze the provided video frames.

Rules:
- List only important events needed to understand the video.
- Keep events in chronological order.
- Use short and clear descriptions.
- Every event must include a start and end timestamp.
- Use timestamp format like 00:00 - 00:20.
- Do not write the letters MM or SS in the timestamp.
- Do not describe every frame separately.
- Do not include explanations.
- Use this exact output format:

Salient event 1: <event description>, 00:00 - 00:20
Salient event 2: <event description>, 00:21 - 00:40
Salient event 3: <event description>, 00:41 - 01:00
"""


REASONING_GUIDED_PROMPT = """
You are given sampled frames from a video in chronological order.
Each frame is labeled with its timestamp.

Task:
Retrieve the salient events of the video with temporal localization.

Before producing the final answer, internally check:
1. Which visible changes are important for understanding the video?
2. Which events are distinct from each other?
3. Which repeated moments should be merged into one event?
4. Which timestamp range best matches each event?
5. Is every timestamp written correctly as 00:00 - 00:20?

Only output the final event list. Do not output your reasoning.

Rules:
- List only salient events needed to understand the video.
- Merge repeated or very similar moments into one event.
- Keep events in chronological order.
- Every event must include a start and end timestamp.
- Use timestamp format like 00:00 - 00:20.
- Do not write the letters MM or SS in the timestamp.
- Do not describe every frame separately.
- Do not include explanations.
- Use this exact output format:

Salient event 1: <event description>, 00:00 - 00:20
Salient event 2: <event description>, 00:21 - 00:40
Salient event 3: <event description>, 00:41 - 01:00
"""

In [ ]:
PROMPT_EXPERIMENTS = {
    "few_shot": FEW_SHOT_PROMPT,
    "reasoning_guided": REASONING_GUIDED_PROMPT
}

In [ ]:
prompt_history = {
    "zero_shot_event_only": EVENT_ONLY_PROMPT,
    "zero_shot_event_timestamp": EVENT_TIMESTAMP_PROMPT,
    "few_shot": FEW_SHOT_PROMPT,
    "reasoning_guided": REASONING_GUIDED_PROMPT
}

prompt_history_path = output_dir(OUTPUT_DIR / "prompt_experiments" / "prompt_history.txt")

with open(prompt_history_path, "w", encoding="utf-8") as f:
    for name, prompt in prompt_history.items():
        f.write("=" * 80 + "\n")
        f.write(f"{name}\n")
        f.write("=" * 80 + "\n")
        f.write(prompt.strip() + "\n\n")

print(f"Prompt history saved to: {prompt_history_path}")

Prompt history saved to: outputs_vlm_event_detection\prompt_experiments\prompt_history.txt


In [ ]:
def run_prompt_experiment(prompt_name, prompt_text):
    """
    Run one prompt strategy over all videos.

    For each video:
    1. sample frames
    2. run the VLM with the selected prompt
    3. save the raw output
    4. parse the timestamped events
    5. save the parsed output

    Returns:
    - experiment_results dictionary
    - combined parsed DataFrame for all videos
    """

    print(f"\n======================================")
    print(f"Running prompt experiment: {prompt_name}")
    print(f"======================================")

    experiment_output_dir = output_dir(OUTPUT_DIR / "prompt_experiments" / prompt_name)
    
    experiment_results = {}
    all_prompt_timestamp_dfs = []

    for video_name, video_path in VIDEOS.items():
        print(f"\nProcessing {video_name} with prompt: {prompt_name}")

        sampled_frames_df, vlm_output, parsed_df = run_and_parse_pipeline(
            video_name=video_name,
            video_path=video_path,
            output_dir=experiment_output_dir,
            prompt_name=prompt_name,
            prompt_text=prompt_text,
            parser_func=parse_event_with_timestamps 
        )

        all_prompt_timestamp_dfs.append(parsed_df)

        experiment_results[video_name] = {
            "sampled_frames": sampled_frames_df,
            "raw_output": vlm_output,
            "parsed_timestamp": parsed_df
        }

        print("\nParsed output:")

    combined_df = pd.concat(all_prompt_timestamp_dfs, ignore_index=True)
    combined_df.to_csv(experiment_output_dir / "all_videos_parsed_timestamp.csv", index=False)

    print(f"\nFinished prompt experiment: {prompt_name}")
    if PLOT:
        display(combined_df)

    return experiment_results, combined_df

In [ ]:
all_prompt_results = {}
all_prompt_parsed_dfs = []

for prompt_name, prompt_text in PROMPT_EXPERIMENTS.items():
    results, combined_df = run_prompt_experiment(prompt_name, prompt_text)

    all_prompt_results[prompt_name] = results
    all_prompt_parsed_dfs.append(combined_df)

new_prompt_outputs_df = pd.concat(all_prompt_parsed_dfs, ignore_index=True)
if PLOT:
    display(new_prompt_outputs_df)

new_prompt_outputs_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "fewshot_reasoning_outputs.csv",
    index=False
)


Running prompt experiment: few_shot

Processing TVSUM_9 with prompt: few_shot
Sampled 12 frames.

Parsed output:


,video_name,prompt_name,event_id,event_description,start,end,start_seconds,end_seconds
0,TVSUM_9,few_shot,1,A person sits next to a car,00:00,00:20,0,20
1,TVSUM_9,few_shot,2,The person looks around,00:21,00:40,21,40
2,TVSUM_9,few_shot,3,The car drives on the road,00:41,01:00,41,60
3,TVSUM_9,few_shot,4,The car's interior is shown,01:01,01:20,61,80
4,TVSUM_9,few_shot,5,The person looks at the camera,01:21,01:40,81,100
5,TVSUM_9,few_shot,6,The car drives on the road again,01:41,02:00,101,120
6,TVSUM_9,few_shot,7,The car's interior is shown again,02:01,02:20,121,140
7,TVSUM_9,few_shot,8,The car drives on the road once more,02:21,02:40,141,160
8,TVSUM_9,few_shot,9,A woman drives the car,02:41,03:00,161,180
9,TVSUM_9,few_shot,10,The car drives on the road again,03:01,03:20,181,200



Processing TVSUM_10 with prompt: few_shot
Sampled 12 frames.

Parsed output:


,video_name,prompt_name,event_id,event_description,start,end,start_seconds,end_seconds
0,TVSUM_10,few_shot,1,An electric car is shown being charged,00:00,00:10,0,10
1,TVSUM_10,few_shot,2,A news anchor introduces an electric car segment,00:11,00:20,11,20
2,TVSUM_10,few_shot,3,A man discusses the charging system for electr...,00:21,00:30,21,30
3,TVSUM_10,few_shot,4,A graph showing data related to electric cars ...,00:31,00:40,31,40
4,TVSUM_10,few_shot,5,A man speaks about the benefits of electric cars,00:41,00:50,41,50
5,TVSUM_10,few_shot,6,A man discusses the importance of electric car...,00:51,01:00,51,60
6,TVSUM_10,few_shot,7,A bus is shown driving on a road,01:01,01:10,61,70
7,TVSUM_10,few_shot,8,Attendees are shown taking notes during a meeting,01:11,01:20,71,80
8,TVSUM_10,few_shot,9,Attendees are shown interacting with a display...,01:21,01:30,81,90
9,TVSUM_10,few_shot,10,A man speaks about the future of electric cars,01:31,01:40,91,100



Processing TVSUM_11 with prompt: few_shot
Sampled 12 frames.

Parsed output:


...


Processing TVSUM_12 with prompt: reasoning_guided
Sampled 12 frames.

Parsed output:


,video_name,prompt_name,event_id,event_description,start,end,start_seconds,end_seconds
0,TVSUM_12,reasoning_guided,1,A person is washing their hands in a sink,00:00,00:20,0,20
1,TVSUM_12,reasoning_guided,2,The scene transitions to a woman talking in a ...,00:21,00:40,21,40
2,TVSUM_12,reasoning_guided,3,The woman is shown holding a bottle of thicken...,00:41,01:00,41,60
3,TVSUM_12,reasoning_guided,4,The woman is seen applying the thickening agen...,01:01,01:20,61,80
4,TVSUM_12,reasoning_guided,5,The woman is shown in an elevator,01:21,01:40,81,100
5,TVSUM_12,reasoning_guided,6,"The woman is outside, waving at the camera",01:41,01:50,101,110
6,TVSUM_12,reasoning_guided,7,"The woman is back in the kitchen, talking to s...",01:51,02:00,111,120
7,TVSUM_12,reasoning_guided,8,"The woman is shown in a bathroom, washing her ...",02:01,02:20,121,140
8,TVSUM_12,reasoning_guided,9,"The woman is shown in a living room, playing w...",02:21,02:40,141,160
9,TVSUM_12,reasoning_guided,10,"The woman is shown in a bathroom, washing her ...",02:41,03:20,161,200



Finished prompt experiment: reasoning_guided


,video_name,prompt_name,event_id,event_description,start,end,start_seconds,end_seconds
0,TVSUM_9,reasoning_guided,1,A man in glasses and a black sweater is talkin...,00:00,00:20,0,20
1,TVSUM_9,reasoning_guided,2,The scene transitions to a futuristic-looking ...,00:21,00:40,21,40
2,TVSUM_9,reasoning_guided,3,"The interior of the vehicle is shown, highligh...",00:41,01:00,41,60
3,TVSUM_9,reasoning_guided,4,"The man continues his conversation, possibly e...",01:01,01:20,61,80
4,TVSUM_9,reasoning_guided,5,"The futuristic vehicle is shown again, emphasi...",01:21,01:40,81,100
5,TVSUM_9,reasoning_guided,6,"The man returns to the camera, continuing his ...",01:41,02:00,101,120
6,TVSUM_9,reasoning_guided,7,"The futuristic vehicle is displayed again, foc...",02:01,02:20,121,140
7,TVSUM_9,reasoning_guided,8,"The man is seen inside the vehicle, demonstrat...",02:21,02:40,141,160
8,TVSUM_9,reasoning_guided,9,A woman is shown driving the futuristic vehicl...,02:41,03:00,161,180
9,TVSUM_9,reasoning_guided,10,The video concludes with an advertisement for ...,03:01,03:20,181,200


,video_name,prompt_name,event_id,event_description,start,end,start_seconds,end_seconds
0,TVSUM_9,few_shot,1,A person sits next to a car,00:00,00:20,0,20
1,TVSUM_9,few_shot,2,The person looks around,00:21,00:40,21,40
2,TVSUM_9,few_shot,3,The car drives on the road,00:41,01:00,41,60
3,TVSUM_9,few_shot,4,The car's interior is shown,01:01,01:20,61,80
4,TVSUM_9,few_shot,5,The person looks at the camera,01:21,01:40,81,100
...,...,...,...,...,...,...,...,...
110,TVSUM_12,reasoning_guided,11,"The woman is shown in a living room, playing w...",03:21,03:40,201,220
111,TVSUM_12,reasoning_guided,12,"The woman is shown in a bathroom, washing her ...",03:41,04:00,221,240
112,TVSUM_12,reasoning_guided,13,"The woman is shown in a living room, playing w...",04:01,04:20,241,260
113,TVSUM_12,reasoning_guided,14,"The woman is shown in a bathroom, washing her ...",04:21,04:40,261,280


In [ ]:
SIMILARITY_THRESHOLD = 0.45

prompt_evaluation_dfs = []
prompt_summary_rows = []

for prompt_name in PROMPT_EXPERIMENTS.keys():
    print(f"\nEvaluating prompt: {prompt_name}")

    prompt_vlm_df = new_prompt_outputs_df[
        new_prompt_outputs_df["prompt_name"] == prompt_name
    ].copy()

    event_similarity_df_prompt = compute_event_similarity_matches(
        manual_df=manual_annotations_df,
        vlm_df=prompt_vlm_df,
        similarity_threshold=SIMILARITY_THRESHOLD
    )

    event_similarity_df_prompt["prompt_name"] = prompt_name

    prompt_evaluation_dfs.append(event_similarity_df_prompt)

    for video_name, group_df in event_similarity_df_prompt.groupby("video_name"):
        total_manual_events = len(group_df)
        matched_events = int(group_df["matched"].sum())
        missed_events = total_manual_events - matched_events

        prompt_summary_rows.append({
            "prompt_name": prompt_name,
            "video_name": video_name,
            "manual_events": total_manual_events,
            "matched_events": matched_events,
            "missed_events": missed_events,
            "match_rate": matched_events / total_manual_events if total_manual_events > 0 else 0
        })

new_prompt_evaluations_df = pd.concat(prompt_evaluation_dfs, ignore_index=True)
new_prompt_summary_df = pd.DataFrame(prompt_summary_rows)
if PLOT:
    display(new_prompt_evaluations_df.head())
    display(new_prompt_summary_df.head())

new_prompt_evaluations_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "fewshot_reasoning_evaluations.csv",
    index=False
)

new_prompt_summary_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "fewshot_reasoning_summary.csv",
    index=False
)


Evaluating prompt: few_shot

Evaluating prompt: reasoning_guided


,video_name,manual_event_id,manual_event,manual_start,manual_end,subjectivity,best_vlm_event_id,best_vlm_event,similarity,matched,prompt_name
0,TVSUM_9,1,Man gets into enclosed two-wheeled vehicle,00:00,00:06,2,4,The car's interior is shown,0.326225,False,few_shot
1,TVSUM_9,2,Man speaking to camera,00:14,00:20,1,5,The person looks at the camera,0.569412,True,few_shot
2,TVSUM_9,3,Man gets out of enclosed two-wheeled vehicle,00:26,00:32,2,11,The car is shown with a diagram,0.320805,False,few_shot
3,TVSUM_9,4,Man speaking to camera,00:33,00:42,-1,5,The person looks at the camera,0.569412,True,few_shot
4,TVSUM_9,5,Functionality of the vehicle is explained,00:43,00:57,2,11,The car is shown with a diagram,0.632571,True,few_shot
...,...,...,...,...,...,...,...,...,...,...,...
69,TVSUM_12,3,A dog getting pet on a kitchen countertop,02:29,02:56,1,9,"The woman is shown in a living room, playing w...",0.442174,False,reasoning_guided
70,TVSUM_12,4,A dog getting pet on a kitchen countertop with...,04:25,04:30,2,9,"The woman is shown in a living room, playing w...",0.335740,False,reasoning_guided
71,TVSUM_12,5,An electric razor being shown next to a dog,04:31,04:34,2,9,"The woman is shown in a living room, playing w...",0.264296,False,reasoning_guided
72,TVSUM_12,6,Hair cutting materials being shown with a pile...,05:41,05:47,2,4,The woman is seen applying the thickening agen...,0.406852,False,reasoning_guided


,prompt_name,video_name,manual_events,matched_events,missed_events,match_rate
0,few_shot,TVSUM_10,6,3,3,0.500000
1,few_shot,TVSUM_11,8,7,1,0.875000
2,few_shot,TVSUM_12,7,2,5,0.285714
3,few_shot,TVSUM_9,16,7,9,0.437500
4,reasoning_guided,TVSUM_10,6,1,5,0.166667
5,reasoning_guided,TVSUM_11,8,7,1,0.875000
6,reasoning_guided,TVSUM_12,7,2,5,0.285714
7,reasoning_guided,TVSUM_9,16,9,7,0.562500


In [ ]:
zero_shot_eval_df = compute_event_similarity_matches(
    manual_df=manual_annotations_df,
    vlm_df=all_timestamp_df,
    similarity_threshold=SIMILARITY_THRESHOLD
)

zero_shot_eval_df["prompt_name"] = "zero_shot"

zero_shot_summary_rows = []

for video_name, group_df in zero_shot_eval_df.groupby("video_name"):
    total_manual_events = len(group_df)
    matched_events = int(group_df["matched"].sum())
    missed_events = total_manual_events - matched_events

    zero_shot_summary_rows.append({
        "prompt_name": "zero_shot",
        "video_name": video_name,
        "manual_events": total_manual_events,
        "matched_events": matched_events,
        "missed_events": missed_events,
        "match_rate": matched_events / total_manual_events if total_manual_events > 0 else 0
    })

zero_shot_summary_df = pd.DataFrame(zero_shot_summary_rows)
if PLOT:
    display(zero_shot_eval_df.head())
    display(zero_shot_summary_df.head())

zero_shot_eval_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "zero_shot_evaluation.csv",
    index=False
)

zero_shot_summary_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "zero_shot_summary.csv",
    index=False
)

# Missed events
new_prompt_missed_events_df = new_prompt_evaluations_df[
    new_prompt_evaluations_df["matched"] == False
].copy()

new_prompt_missed_events_df = new_prompt_missed_events_df.sort_values(
    by=["prompt_name", "video_name", "subjectivity", "similarity"],
    ascending=[True, True, False, True]
)

new_prompt_missed_events_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "fewshot_reasoning_missed_events.csv",
    index=False
)
if PLOT:
    display(new_prompt_missed_events_df.head())
    display(evaluation_summary_df.head())

evaluation_summary_df.to_csv(
    OUTPUT_DIR / "evaluation_summary.csv",
    index=False
)

,video_name,manual_event_id,manual_event,manual_start,manual_end,subjectivity,best_vlm_event_id,best_vlm_event,similarity,matched,prompt_name
0,TVSUM_9,1,Man gets into enclosed two-wheeled vehicle,00:00,00:06,2,4,Man sitting next to a car again,0.409412,False,zero_shot
1,TVSUM_9,2,Man speaking to camera,00:14,00:20,1,1,"Man sitting next to a car, talking",0.481796,True,zero_shot
2,TVSUM_9,3,Man gets out of enclosed two-wheeled vehicle,00:26,00:32,2,4,Man sitting next to a car again,0.414626,False,zero_shot
3,TVSUM_9,4,Man speaking to camera,00:33,00:42,-1,1,"Man sitting next to a car, talking",0.481796,True,zero_shot
4,TVSUM_9,5,Functionality of the vehicle is explained,00:43,00:57,2,9,Close-up of car components,0.437063,False,zero_shot
5,TVSUM_9,6,Vehicle simulation shown,00:58,01:02,2,2,Car driving on a road,0.399613,False,zero_shot
6,TVSUM_9,7,Man speaking to camera,01:02,01:06,-1,1,"Man sitting next to a car, talking",0.481796,True,zero_shot
7,TVSUM_9,8,Simulation of the inner workings of the two-wh...,01:09,01:25,2,9,Close-up of car components,0.267351,False,zero_shot
8,TVSUM_9,9,Pedal controls shown,01:25,01:30,1,6,Close-up of car pedals again,0.553858,True,zero_shot
9,TVSUM_9,10,Smart display shows vehicle data,01:59,02:03,1,4,Man sitting next to a car again,0.277188,False,zero_shot


,prompt_name,video_name,manual_events,matched_events,missed_events,match_rate
0,zero_shot,TVSUM_10,6,1,5,0.166667
1,zero_shot,TVSUM_11,8,7,1,0.875000
2,zero_shot,TVSUM_12,7,2,5,0.285714
3,zero_shot,TVSUM_9,16,4,12,0.250000


,video_name,manual_event_id,manual_event,manual_start,manual_end,subjectivity,best_vlm_event_id,best_vlm_event,similarity,matched,prompt_name
16,TVSUM_10,1,Woman talking on a set,00:05,00:25,1,11,A woman demonstrates how to use a charging sta...,0.298770,False,few_shot
19,TVSUM_10,4,Man talking,00:46,01:03,0,10,A man speaks about the future of electric cars,0.415952,False,few_shot
17,TVSUM_10,2,People looking at a car,00:26,00:28,0,1,An electric car is shown being charged,0.446791,False,few_shot
25,TVSUM_11,4,A dog getting dried,01:30,01:32,2,2,The person interacts with a dog,0.396245,False,few_shot
33,TVSUM_12,4,A dog getting pet on a kitchen countertop with...,04:25,04:30,2,6,The person applies hair conditioner to their hair,0.189632,False,few_shot
34,TVSUM_12,5,An electric razor being shown next to a dog,04:31,04:34,2,9,The person applies hair gel to their hair,0.326299,False,few_shot
32,TVSUM_12,3,A dog getting pet on a kitchen countertop,02:29,02:56,1,6,The person applies hair conditioner to their hair,0.128142,False,few_shot
31,TVSUM_12,2,A dog playing around a kitchen countertop,01:50,02:05,0,1,A person washes their hands,0.096278,False,few_shot
30,TVSUM_12,1,A woman filming herself walking and talking ar...,01:08,01:15,-1,10,The person applies hair spray to their hair,0.131821,False,few_shot
11,TVSUM_9,12,Battery layout and controls showcased,02:41,02:50,2,17,The car is shown with a sixth diagram,0.245520,False,few_shot


,video_name,manual_events,matched_events,missed_events,match_rate
0,TVSUM_10,6,1,5,0.166667
1,TVSUM_11,8,7,1,0.875000
2,TVSUM_12,7,2,5,0.285714
3,TVSUM_9,16,4,12,0.250000


In [ ]:

zero_shot_missed_events_df = zero_shot_eval_df[
    zero_shot_eval_df["matched"] == False
].copy()

zero_shot_missed_events_df["prompt_name"] = "zero_shot"

zero_shot_missed_events_df = zero_shot_missed_events_df.sort_values(
    by=["prompt_name", "video_name", "subjectivity", "similarity"],
    ascending=[True, True, False, True]
)
if PLOT:
    display(zero_shot_missed_events_df.head())

zero_shot_missed_events_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "zero_shot_missed_events.csv",
    index=False
)

,video_name,manual_event_id,manual_event,manual_start,manual_end,subjectivity,best_vlm_event_id,best_vlm_event,similarity,matched,prompt_name
21,TVSUM_10,6,Man talking with cars in the background,01:28,01:42,2,5,Overview of the electric vehicle industry,0.223952,False,zero_shot
18,TVSUM_10,3,Cable being plugged into a car,00:32,00:33,2,3,Demonstration of the charging process for elec...,0.369192,False,zero_shot
16,TVSUM_10,1,Woman talking on a set,00:05,00:25,1,9,Attendees taking notes during a presentation o...,0.294872,False,zero_shot
19,TVSUM_10,4,Man talking,00:46,01:03,0,9,Attendees taking notes during a presentation o...,0.262039,False,zero_shot
17,TVSUM_10,2,People looking at a car,00:26,00:28,0,3,Demonstration of the charging process for elec...,0.273575,False,zero_shot
25,TVSUM_11,4,A dog getting dried,01:30,01:32,2,2,Woman preparing a dog for grooming,0.341485,False,zero_shot
36,TVSUM_12,7,A pair of hair clippers being shown,06:34,06:40,2,3,Woman showing products,0.285827,False,zero_shot
35,TVSUM_12,6,Hair cutting materials being shown with a pile...,05:41,05:47,2,1,Person washing dog's fur,0.290694,False,zero_shot
34,TVSUM_12,5,An electric razor being shown next to a dog,04:31,04:34,2,1,Person washing dog's fur,0.400482,False,zero_shot
33,TVSUM_12,4,A dog getting pet on a kitchen countertop with...,04:25,04:30,2,1,Person washing dog's fur,0.426629,False,zero_shot


# Prompt Comparison Summary

The results from the zero-shot, few-shot, and reasoning-guided prompts are combined into one summary table.

This makes it easier to compare the overall match rate of each prompting strategy.

In [ ]:
all_prompt_summary_df = pd.concat(
    [zero_shot_summary_df, new_prompt_summary_df],
    ignore_index=True
)
if PLOT:
    display(all_prompt_summary_df)

all_prompt_summary_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "all_prompt_summary.csv",
    index=False
)

,prompt_name,video_name,manual_events,matched_events,missed_events,match_rate
0,zero_shot,TVSUM_10,6,1,5,0.166667
1,zero_shot,TVSUM_11,8,7,1,0.875000
2,zero_shot,TVSUM_12,7,2,5,0.285714
3,zero_shot,TVSUM_9,16,4,12,0.250000
4,few_shot,TVSUM_10,6,3,3,0.500000
5,few_shot,TVSUM_11,8,7,1,0.875000
6,few_shot,TVSUM_12,7,2,5,0.285714
7,few_shot,TVSUM_9,16,7,9,0.437500
8,reasoning_guided,TVSUM_10,6,1,5,0.166667
9,reasoning_guided,TVSUM_11,8,7,1,0.875000


In [ ]:
all_prompt_missed_events_df = pd.concat(
    [zero_shot_missed_events_df, new_prompt_missed_events_df],
    ignore_index=True
)

all_prompt_missed_events_df = all_prompt_missed_events_df.sort_values(
    by=["prompt_name", "video_name", "subjectivity", "similarity"],
    ascending=[True, True, False, True]
)
if PLOT:
    display(all_prompt_missed_events_df.head())

all_prompt_missed_events_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "all_prompt_missed_events.csv",
    index=False
)

,video_name,manual_event_id,manual_event,manual_start,manual_end,subjectivity,best_vlm_event_id,best_vlm_event,similarity,matched,prompt_name
23,TVSUM_10,1,Woman talking on a set,00:05,00:25,1,11,A woman demonstrates how to use a charging sta...,0.298770,False,few_shot
24,TVSUM_10,4,Man talking,00:46,01:03,0,10,A man speaks about the future of electric cars,0.415952,False,few_shot
25,TVSUM_10,2,People looking at a car,00:26,00:28,0,1,An electric car is shown being charged,0.446791,False,few_shot
26,TVSUM_11,4,A dog getting dried,01:30,01:32,2,2,The person interacts with a dog,0.396245,False,few_shot
27,TVSUM_12,4,A dog getting pet on a kitchen countertop with...,04:25,04:30,2,6,The person applies hair conditioner to their hair,0.189632,False,few_shot
28,TVSUM_12,5,An electric razor being shown next to a dog,04:31,04:34,2,9,The person applies hair gel to their hair,0.326299,False,few_shot
29,TVSUM_12,3,A dog getting pet on a kitchen countertop,02:29,02:56,1,6,The person applies hair conditioner to their hair,0.128142,False,few_shot
30,TVSUM_12,2,A dog playing around a kitchen countertop,01:50,02:05,0,1,A person washes their hands,0.096278,False,few_shot
31,TVSUM_12,1,A woman filming herself walking and talking ar...,01:08,01:15,-1,10,The person applies hair spray to their hair,0.131821,False,few_shot
32,TVSUM_9,12,Battery layout and controls showcased,02:41,02:50,2,17,The car is shown with a sixth diagram,0.245520,False,few_shot


In [ ]:
miss_summary_by_prompt_df = (
    all_prompt_missed_events_df
    .groupby(["prompt_name", "video_name"])
    .size()
    .reset_index(name="missed_count")
)

display(miss_summary_by_prompt_df)

miss_summary_by_prompt_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "miss_summary_by_prompt.csv",
    index=False
)

,prompt_name,video_name,missed_count
0,few_shot,TVSUM_10,3
1,few_shot,TVSUM_11,1
2,few_shot,TVSUM_12,5
3,few_shot,TVSUM_9,9
4,reasoning_guided,TVSUM_10,5
5,reasoning_guided,TVSUM_11,1
6,reasoning_guided,TVSUM_12,5
7,reasoning_guided,TVSUM_9,7
8,zero_shot,TVSUM_10,5
9,zero_shot,TVSUM_11,1


In [ ]:
average_prompt_performance_df = (
    all_prompt_summary_df
    .groupby("prompt_name")
    .agg(
        total_manual_events=("manual_events", "sum"),
        total_matched_events=("matched_events", "sum"),
        total_missed_events=("missed_events", "sum")
    )
    .reset_index()
)

average_prompt_performance_df["overall_match_rate"] = (
    average_prompt_performance_df["total_matched_events"] /
    average_prompt_performance_df["total_manual_events"]
)

average_prompt_performance_df = average_prompt_performance_df.sort_values(
    by="overall_match_rate",
    ascending=False
)

display(average_prompt_performance_df)

average_prompt_performance_df.to_csv(
    OUTPUT_DIR / "prompt_experiments" / "average_prompt_performance.csv",
    index=False
)

,prompt_name,total_manual_events,total_matched_events,total_missed_events,overall_match_rate
0,few_shot,37,19,18,0.513514
1,reasoning_guided,37,19,18,0.513514
2,zero_shot,37,14,23,0.378378


In [ ]:
# Best prompt
best_prompt = average_prompt_performance_df.iloc[0]

print("Best prompt strategy:")
print(f"Prompt: {best_prompt['prompt_name']}")
print(f"Overall match rate: {best_prompt['overall_match_rate']:.2f}")
print(
    f"Matched events: "
    f"{best_prompt['total_matched_events']} / {best_prompt['total_manual_events']}"
)

Best prompt strategy:
Prompt: few_shot
Overall match rate: 0.51
Matched events: 19 / 37


# Part 4: Moment Retrieval Using Pre-trained Models

In this section, we use the detected events from Task 3 as text queries for moment retrieval.

The goal is to retrieve the video segment that best matches each event and predict:
- start timestamp,
- end timestamp,
- confidence score.

We use two pretrained moment retrieval models:

1. **CG-DETR through Lighthouse with CLIP features**  
   CG-DETR is a dedicated moment retrieval model loaded through the Lighthouse framework. We use the pretrained `clip_cg_detr_qvhighlight.ckpt` checkpoint from the Lighthouse pretrained weights: https://zenodo.org/records/13363606. The model is initialized with `feature_name="clip"`, meaning that Lighthouse uses CLIP-based video-text features for retrieval. Given a video chunk and a text query, CG-DETR predicts the most relevant temporal window and a confidence score.

2. **Moment-DETR with CLIP features**  
   Moment-DETR is also a dedicated moment retrieval model. We use the pretrained `model_best.ckpt` checkpoint, which uses CLIP image/text features inside the official inference pipeline.

CLIP is not used as a standalone baseline. It is used as the feature extractor inside both retrieval models.

Because both pretrained models have a video length limit of about 150 seconds, we split long TVSUM videos into 120-second chunks. Each query is run over all chunks, and the highest-confidence prediction is kept.

In [ ]:
TASK3_OUTPUT_DIR = Path("outputs_vlm_event_detection")
MOMENT_OUTPUT_DIR = output_dir("outputs_moment_retrieval")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
print("Current working directory:", Path.cwd())

for video_name, video_path in VIDEOS.items():
    print(video_name, "->", video_path, "| exists:", video_path.exists())

Using device: cuda
Current working directory: c:\Users\aggel\OneDrive\Υπολογιστής\Comp Vision\cv_grp3_2026
TVSUM_9 -> videos\video_9.mp4 | exists: True
TVSUM_10 -> videos\video_10.mp4 | exists: True
TVSUM_11 -> videos\video_11.mp4 | exists: True
TVSUM_12 -> videos\video_12.mp4 | exists: True


In [ ]:
SELECTED_PROMPT_FOR_RETRIEVAL = "few_shot"

TASK3_PROMPT_OUTPUTS_PATH = (
    TASK3_OUTPUT_DIR / "prompt_experiments" / "fewshot_reasoning_outputs.csv"
)

if not TASK3_PROMPT_OUTPUTS_PATH.exists():
    raise FileNotFoundError(
        f"Task 3 prompt output file not found: {TASK3_PROMPT_OUTPUTS_PATH}"
    )

task3_outputs_df = pd.read_csv(TASK3_PROMPT_OUTPUTS_PATH)

vlm_queries_df = task3_outputs_df[
    task3_outputs_df["prompt_name"] == SELECTED_PROMPT_FOR_RETRIEVAL
].copy()

print(f"Loaded Task 3 prompt outputs from: {TASK3_PROMPT_OUTPUTS_PATH}")
print(f"Using prompt: {SELECTED_PROMPT_FOR_RETRIEVAL}")
print(f"Number of retrieval queries: {len(vlm_queries_df)}")

display(vlm_queries_df.head())

Loaded Task 3 prompt outputs from: outputs_vlm_event_detection\prompt_experiments\fewshot_reasoning_outputs.csv
Using prompt: few_shot
Number of retrieval queries: 68


,video_name,prompt_name,event_id,event_description,start,end,start_seconds,end_seconds
0,TVSUM_9,few_shot,1,A person sits next to a car,00:00,00:20,0,20
1,TVSUM_9,few_shot,2,The person looks around,00:21,00:40,21,40
2,TVSUM_9,few_shot,3,The car drives on the road,00:41,01:00,41,60
3,TVSUM_9,few_shot,4,The car's interior is shown,01:01,01:20,61,80
4,TVSUM_9,few_shot,5,The person looks at the camera,01:21,01:40,81,100


In [ ]:
def expand_query(base_query):
    base_query = str(base_query).strip()

    variations = [
        base_query,
        f"a video of {base_query}",
        f"the moment when {base_query}",
        f"a scene where {base_query}",
        f"someone is involved in: {base_query}",
    ]

    unique_variations = []

    for query in variations:
        if query not in unique_variations:
            unique_variations.append(query)

    return unique_variations

In [ ]:
def get_video_duration_seconds(video_path):
    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    cap.release()

    if fps == 0:
        raise ValueError(f"Could not read FPS from video: {video_path}")

    return frame_count / fps


def create_video_chunk(video_path, output_path, start_seconds, end_seconds):
    video_path = Path(video_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists():
        return output_path

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if fps == 0:
        cap.release()
        raise ValueError(f"Could not read FPS from video: {video_path}")

    start_frame = int(start_seconds * fps)
    end_frame = int(end_seconds * fps)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        str(output_path),
        fourcc,
        fps,
        (width, height)
    )

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    current_frame = start_frame

    while current_frame < end_frame:
        success, frame = cap.read()

        if not success:
            break

        writer.write(frame)
        current_frame += 1

    cap.release()
    writer.release()

    return output_path

In [ ]:
LIGHTHOUSE_REPO_DIR = Path("lighthouse").resolve()

print("Lighthouse repo exists:", LIGHTHOUSE_REPO_DIR.exists())
print("models.py exists:", (LIGHTHOUSE_REPO_DIR / "lighthouse" / "models.py").exists())

if not (LIGHTHOUSE_REPO_DIR / "lighthouse" / "models.py").exists():
    raise FileNotFoundError(
        "Lighthouse package not found. Make sure the lighthouse folder is inside the repo root."
    )

for name in list(sys.modules.keys()):
    if name == "lighthouse" or name.startswith("lighthouse."):
        del sys.modules[name]

if str(LIGHTHOUSE_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(LIGHTHOUSE_REPO_DIR))

importlib.invalidate_caches()

from lighthouse.models import CGDETRPredictor

CGDETR_CKPT_PATH = Path("clip_cg_detr_qvhighlight.ckpt")

print("CG-DETR checkpoint exists:", CGDETR_CKPT_PATH.exists())
print("CG-DETR checkpoint path:", CGDETR_CKPT_PATH.resolve())

if not CGDETR_CKPT_PATH.exists():
    raise FileNotFoundError(
        "CG-DETR checkpoint is missing. Place it at clip_cg_detr_qvhighlight.ckpt"
    )

cgdetr_model = CGDETRPredictor(
    str(CGDETR_CKPT_PATH),
    device="cpu",
    feature_name="clip"
)

print("CG-DETR model loaded.")

Lighthouse repo exists: True
models.py exists: True
CG-DETR checkpoint exists: True
CG-DETR checkpoint path: C:\Users\aggel\OneDrive\Υπολογιστής\Comp Vision\cv_grp3_2026\clip_cg_detr_qvhighlight.ckpt
CG-DETR model loaded.


In [ ]:
CGDETR_CHUNK_SECONDS = 120
CGDETR_CHUNK_DIR = output_dir(MOMENT_OUTPUT_DIR / "cgdetr_video_chunks")

def make_cgdetr_video_chunks(video_name, video_path, chunk_seconds=120):
    duration = get_video_duration_seconds(video_path)

    chunks = []
    chunk_start = 0.0
    chunk_id = 0

    while chunk_start < duration:
        chunk_end = min(chunk_start + chunk_seconds, duration)

        chunk_path = (
            CGDETR_CHUNK_DIR /
            f"{video_name}_chunk_{chunk_id:03d}_{int(chunk_start)}_{int(chunk_end)}.mp4"
        )

        create_video_chunk(
            video_path=video_path,
            output_path=chunk_path,
            start_seconds=chunk_start,
            end_seconds=chunk_end
        )

        chunks.append({
            "chunk_id": chunk_id,
            "chunk_path": chunk_path,
            "chunk_start": chunk_start,
            "chunk_end": chunk_end
        })

        chunk_start += chunk_seconds
        chunk_id += 1

    return chunks

In [ ]:
def run_cgdetr_on_vlm_events_chunked(model, videos, vlm_events_df):
    rows = []

    for video_name, video_path in videos.items():
        print(f"\n==============================")
        print(f"CG-DETR processing {video_name}")
        print(f"==============================")

        if not video_path.exists():
            print(f"Skipping {video_name}: video file not found.")
            continue

        video_events_df = vlm_events_df[
            vlm_events_df["video_name"] == video_name
        ].copy()

        if video_events_df.empty:
            print(f"No Task 3 events found for {video_name}.")
            continue

        chunks = make_cgdetr_video_chunks(
            video_name=video_name,
            video_path=video_path,
            chunk_seconds=CGDETR_CHUNK_SECONDS
        )

        print(f"Created/found {len(chunks)} chunks for {video_name}.")

        encoded_chunks = []

        for chunk in chunks:
            print(
                f"Encoding chunk {chunk['chunk_id']} "
                f"({seconds_to_mmss(chunk['chunk_start'])} - {seconds_to_mmss(chunk['chunk_end'])})"
            )

            encoded_video = model.encode_video(str(chunk["chunk_path"]))

            encoded_chunks.append({
                "chunk_id": chunk["chunk_id"],
                "chunk_start": chunk["chunk_start"],
                "chunk_end": chunk["chunk_end"],
                "encoded_video": encoded_video
            })

        for _, event_row in video_events_df.iterrows():
            base_query = event_row["event_description"]
            query_variations = expand_query(base_query)

            best_query = None
            best_start = None
            best_end = None
            best_score = -1
            best_chunk_id = None

            for encoded_chunk in encoded_chunks:
                chunk_start_offset = encoded_chunk["chunk_start"]

                for query in query_variations:
                    prediction = model.predict(
                        query,
                        encoded_chunk["encoded_video"]
                    )

                    top_window = prediction["pred_relevant_windows"][0]

                    pred_start_chunk = float(top_window[0])
                    pred_end_chunk = float(top_window[1])
                    confidence = float(top_window[2])

                    pred_start_full = chunk_start_offset + pred_start_chunk
                    pred_end_full = chunk_start_offset + pred_end_chunk

                    if confidence > best_score:
                        best_query = query
                        best_start = pred_start_full
                        best_end = pred_end_full
                        best_score = confidence
                        best_chunk_id = encoded_chunk["chunk_id"]

            rows.append({
                "model_name": "CG-DETR",
                "prompt_name": event_row["prompt_name"],
                "video_name": video_name,
                "vlm_event_id": event_row["event_id"],
                "base_query": base_query,
                "query_used": best_query,
                "best_chunk_id": best_chunk_id,
                "pred_start_seconds": best_start,
                "pred_end_seconds": best_end,
                "pred_start": seconds_to_mmss(best_start),
                "pred_end": seconds_to_mmss(best_end),
                "confidence": best_score
            })

    return pd.DataFrame(rows)


cgdetr_predictions_df = run_cgdetr_on_vlm_events_chunked(
    model=cgdetr_model,
    videos=VIDEOS,
    vlm_events_df=vlm_queries_df
)
if PLOT:
    display(cgdetr_predictions_df.head())

cgdetr_predictions_df.to_csv(
    MOMENT_OUTPUT_DIR / "cgdetr_predictions_from_fewshot_vlm_events.csv",
    index=False
)


CG-DETR processing TVSUM_9
Created/found 2 chunks for TVSUM_9.
Encoding chunk 0 (00:00 - 02:00)
Encoding chunk 1 (02:00 - 03:54)

CG-DETR processing TVSUM_10
Created/found 2 chunks for TVSUM_10.
Encoding chunk 0 (00:00 - 02:00)
Encoding chunk 1 (02:00 - 02:13)

CG-DETR processing TVSUM_11
Created/found 2 chunks for TVSUM_11.
Encoding chunk 0 (00:00 - 02:00)
Encoding chunk 1 (02:00 - 02:37)

CG-DETR processing TVSUM_12
Created/found 4 chunks for TVSUM_12.
Encoding chunk 0 (00:00 - 02:00)
Encoding chunk 1 (02:00 - 04:00)
Encoding chunk 2 (04:00 - 06:00)
Encoding chunk 3 (06:00 - 07:31)


,model_name,prompt_name,video_name,vlm_event_id,base_query,query_used,best_chunk_id,pred_start_seconds,pred_end_seconds,pred_start,pred_end,confidence
0,CG-DETR,few_shot,TVSUM_9,1,A person sits next to a car,A person sits next to a car,0,1.5403,20.9329,00:02,00:21,0.9989
1,CG-DETR,few_shot,TVSUM_9,2,The person looks around,the moment when The person looks around,0,72.5142,89.8179,01:13,01:30,0.9996
2,CG-DETR,few_shot,TVSUM_9,3,The car drives on the road,the moment when The car drives on the road,1,213.7529,228.6061,03:34,03:49,0.9882
3,CG-DETR,few_shot,TVSUM_9,4,The car's interior is shown,a scene where The car's interior is shown,0,69.9871,90.2572,01:10,01:30,0.9947
4,CG-DETR,few_shot,TVSUM_9,5,The person looks at the camera,someone is involved in: The person looks at th...,0,71.9306,90.5704,01:12,01:31,0.9993
...,...,...,...,...,...,...,...,...,...,...,...,...
63,CG-DETR,few_shot,TVSUM_12,13,The person applies hair gel to their hair again,a scene where The person applies hair gel to t...,0,6.1588,23.9610,00:06,00:24,0.9986
64,CG-DETR,few_shot,TVSUM_12,14,The person applies hair spray to their hair again,a scene where The person applies hair spray to...,0,8.1836,24.9071,00:08,00:25,0.9990
65,CG-DETR,few_shot,TVSUM_12,15,The person applies hair gel to their hair again,a scene where The person applies hair gel to t...,0,6.1588,23.9610,00:06,00:24,0.9986
66,CG-DETR,few_shot,TVSUM_12,16,The person applies hair spray to their hair again,a scene where The person applies hair spray to...,0,8.1836,24.9071,00:08,00:25,0.9990


In [ ]:
MOMENT_DETR_REPO_DIR = Path("moment_detr").resolve()
MOMENT_DETR_CKPT_PATH = (
    MOMENT_DETR_REPO_DIR / "run_on_video" / "moment_detr_ckpt" / "model_best.ckpt"
)

print("Moment-DETR repo exists:", MOMENT_DETR_REPO_DIR.exists())
print("run_on_video exists:", (MOMENT_DETR_REPO_DIR / "run_on_video" / "run.py").exists())
print("Moment-DETR checkpoint exists:", MOMENT_DETR_CKPT_PATH.exists())
print("Checkpoint path:", MOMENT_DETR_CKPT_PATH.resolve())

if not MOMENT_DETR_REPO_DIR.exists():
    raise FileNotFoundError(
        "Moment-DETR repo not found. Clone it into the repo root with:\n"
        "git clone https://github.com/jayleicn/moment_detr.git"
    )

if not MOMENT_DETR_CKPT_PATH.exists():
    raise FileNotFoundError(
        "Moment-DETR checkpoint not found. Put model_best.ckpt here:\n"
        f"{MOMENT_DETR_CKPT_PATH}"
    )

if str(MOMENT_DETR_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(MOMENT_DETR_REPO_DIR))

importlib.invalidate_caches()

moment_detr_model = MomentDETRPredictor(
    ckpt_path=str(MOMENT_DETR_CKPT_PATH),
    clip_model_name_or_path="ViT-B/32",
    device=DEVICE
)

print("Moment-DETR model loaded.")

Moment-DETR repo exists: True
run_on_video exists: True
Moment-DETR checkpoint exists: True
Checkpoint path: C:\Users\aggel\OneDrive\Υπολογιστής\Comp Vision\cv_grp3_2026\moment_detr\run_on_video\moment_detr_ckpt\model_best.ckpt
Loading feature extractors...
Loading CLIP models
Loading trained Moment-DETR model...
Moment-DETR model loaded.


In [ ]:
MOMENT_DETR_CHUNK_SECONDS = 120
MOMENT_DETR_CHUNK_DIR = output_dir(MOMENT_OUTPUT_DIR / "moment_detr_video_chunks")

def make_moment_detr_video_chunks(video_name, video_path, chunk_seconds=120):
    duration = get_video_duration_seconds(video_path)

    chunks = []
    chunk_start = 0.0
    chunk_id = 0

    while chunk_start < duration:
        chunk_end = min(chunk_start + chunk_seconds, duration)

        chunk_path = (
            MOMENT_DETR_CHUNK_DIR /
            f"{video_name}_chunk_{chunk_id:03d}_{int(chunk_start)}_{int(chunk_end)}.mp4"
        )

        create_video_chunk(
            video_path=video_path,
            output_path=chunk_path,
            start_seconds=chunk_start,
            end_seconds=chunk_end
        )

        chunks.append({
            "chunk_id": chunk_id,
            "chunk_path": chunk_path,
            "chunk_start": chunk_start,
            "chunk_end": chunk_end
        })

        chunk_start += chunk_seconds
        chunk_id += 1

    return chunks

In [ ]:
def run_moment_detr_on_vlm_events_chunked(model, videos, vlm_events_df):
    rows = []

    for video_name, video_path in videos.items():
        print(f"\n==============================")
        print(f"Moment-DETR processing {video_name}")
        print(f"==============================")

        if not video_path.exists():
            print(f"Skipping {video_name}: video file not found.")
            continue

        video_events_df = vlm_events_df[
            vlm_events_df["video_name"] == video_name
        ].copy()

        if video_events_df.empty:
            print(f"No Task 3 events found for {video_name}.")
            continue

        chunks = make_moment_detr_video_chunks(
            video_name=video_name,
            video_path=video_path,
            chunk_seconds=MOMENT_DETR_CHUNK_SECONDS
        )

        print(f"Created/found {len(chunks)} chunks for {video_name}.")

        for _, event_row in video_events_df.iterrows():
            base_query = event_row["event_description"]
            query_variations = expand_query(base_query)

            best_query = None
            best_start = None
            best_end = None
            best_score = -1
            best_chunk_id = None

            for chunk in chunks:
                chunk_path = chunk["chunk_path"]
                chunk_start_offset = chunk["chunk_start"]

                try:
                    predictions = model.localize_moment(
                        video_path=str(chunk_path),
                        query_list=query_variations
                    )
                except AssertionError as e:
                    print(f"Skipping chunk because it is too long: {chunk_path}")
                    print(e)
                    continue

                for query, prediction in zip(query_variations, predictions):
                    pred_windows = prediction["pred_relevant_windows"]

                    if len(pred_windows) == 0:
                        continue

                    top_window = pred_windows[0]

                    pred_start_chunk = float(top_window[0])
                    pred_end_chunk = float(top_window[1])
                    confidence = float(top_window[2])

                    pred_start_full = chunk_start_offset + pred_start_chunk
                    pred_end_full = chunk_start_offset + pred_end_chunk

                    if confidence > best_score:
                        best_query = query
                        best_start = pred_start_full
                        best_end = pred_end_full
                        best_score = confidence
                        best_chunk_id = chunk["chunk_id"]

            if best_start is None:
                continue

            rows.append({
                "model_name": "Moment-DETR",
                "prompt_name": event_row["prompt_name"],
                "video_name": video_name,
                "vlm_event_id": event_row["event_id"],
                "base_query": base_query,
                "query_used": best_query,
                "best_chunk_id": best_chunk_id,
                "pred_start_seconds": best_start,
                "pred_end_seconds": best_end,
                "pred_start": seconds_to_mmss(best_start),
                "pred_end": seconds_to_mmss(best_end),
                "confidence": best_score
            })

    return pd.DataFrame(rows)


moment_detr_predictions_df = run_moment_detr_on_vlm_events_chunked(
    model=moment_detr_model,
    videos=VIDEOS,
    vlm_events_df=vlm_queries_df
)
if PLOT:
    display(moment_detr_predictions_df)

moment_detr_predictions_df.to_csv(
    MOMENT_OUTPUT_DIR / "moment_detr_predictions_from_fewshot_vlm_events.csv",
    index=False
)


Moment-DETR processing TVSUM_9
Created/found 2 chunks for TVSUM_9.

Moment-DETR processing TVSUM_10
Created/found 2 chunks for TVSUM_10.

Moment-DETR processing TVSUM_11
Created/found 2 chunks for TVSUM_11.

Moment-DETR processing TVSUM_12
Created/found 4 chunks for TVSUM_12.


,model_name,prompt_name,video_name,vlm_event_id,base_query,query_used,best_chunk_id,pred_start_seconds,pred_end_seconds,pred_start,pred_end,confidence
0,Moment-DETR,few_shot,TVSUM_9,1,A person sits next to a car,someone is involved in: A person sits next to ...,1,187.0459,224.7556,03:07,03:45,0.9954
1,Moment-DETR,few_shot,TVSUM_9,2,The person looks around,someone is involved in: The person looks around,0,10.5672,39.5695,00:11,00:40,0.9993
2,Moment-DETR,few_shot,TVSUM_9,3,The car drives on the road,a scene where The car drives on the road,1,210.7039,234.2458,03:31,03:54,0.9956
3,Moment-DETR,few_shot,TVSUM_9,4,The car's interior is shown,the moment when The car's interior is shown,0,0.6167,19.7130,00:01,00:20,0.9941
4,Moment-DETR,few_shot,TVSUM_9,5,The person looks at the camera,a video of The person looks at the camera,0,8.2833,42.2595,00:08,00:42,0.9787
...,...,...,...,...,...,...,...,...,...,...,...,...
63,Moment-DETR,few_shot,TVSUM_12,13,The person applies hair gel to their hair again,The person applies hair gel to their hair again,3,395.1354,408.9828,06:35,06:49,0.9990
64,Moment-DETR,few_shot,TVSUM_12,14,The person applies hair spray to their hair again,a scene where The person applies hair spray to...,0,102.4250,120.0150,01:42,02:00,0.9978
65,Moment-DETR,few_shot,TVSUM_12,15,The person applies hair gel to their hair again,someone is involved in: The person applies hai...,0,105.7520,119.9699,01:46,02:00,0.9980
66,Moment-DETR,few_shot,TVSUM_12,16,The person applies hair spray to their hair again,the moment when The person applies hair spray ...,1,119.8766,140.7893,02:00,02:21,0.9989


In [ ]:
all_moment_predictions_df = pd.concat(
    [cgdetr_predictions_df, moment_detr_predictions_df],
    ignore_index=True
)

all_moment_predictions_df = all_moment_predictions_df[
    all_moment_predictions_df["model_name"].isin(["CG-DETR", "Moment-DETR"])
].copy()

if PLOT:
    print(all_moment_predictions_df["model_name"].unique())
    display(all_moment_predictions_df)

all_moment_predictions_df.to_csv(
    MOMENT_OUTPUT_DIR / "all_moment_retrieval_predictions.csv",
    index=False
)

['CG-DETR' 'Moment-DETR']


,model_name,prompt_name,video_name,vlm_event_id,base_query,query_used,best_chunk_id,pred_start_seconds,pred_end_seconds,pred_start,pred_end,confidence
0,CG-DETR,few_shot,TVSUM_9,1,A person sits next to a car,A person sits next to a car,0,1.5403,20.9329,00:02,00:21,0.9989
1,CG-DETR,few_shot,TVSUM_9,2,The person looks around,the moment when The person looks around,0,72.5142,89.8179,01:13,01:30,0.9996
2,CG-DETR,few_shot,TVSUM_9,3,The car drives on the road,the moment when The car drives on the road,1,213.7529,228.6061,03:34,03:49,0.9882
3,CG-DETR,few_shot,TVSUM_9,4,The car's interior is shown,a scene where The car's interior is shown,0,69.9871,90.2572,01:10,01:30,0.9947
4,CG-DETR,few_shot,TVSUM_9,5,The person looks at the camera,someone is involved in: The person looks at th...,0,71.9306,90.5704,01:12,01:31,0.9993
...,...,...,...,...,...,...,...,...,...,...,...,...
131,Moment-DETR,few_shot,TVSUM_12,13,The person applies hair gel to their hair again,The person applies hair gel to their hair again,3,395.1354,408.9828,06:35,06:49,0.9990
132,Moment-DETR,few_shot,TVSUM_12,14,The person applies hair spray to their hair again,a scene where The person applies hair spray to...,0,102.4250,120.0150,01:42,02:00,0.9978
133,Moment-DETR,few_shot,TVSUM_12,15,The person applies hair gel to their hair again,someone is involved in: The person applies hai...,0,105.7520,119.9699,01:46,02:00,0.9980
134,Moment-DETR,few_shot,TVSUM_12,16,The person applies hair spray to their hair again,the moment when The person applies hair spray ...,1,119.8766,140.7893,02:00,02:21,0.9989


# Part 5: Evaluating Moment Retrieval Boundaries

In this section, we evaluate the predicted temporal windows from the moment retrieval models.

The manual annotations are used as ground truth. For each predicted moment, we find the most semantically similar manual annotation from the same video. Then we compute temporal Intersection over Union between the predicted timestamp and the manual timestamp.

The IoU score measures how much the predicted segment overlaps with the annotated segment.

A higher IoU means better temporal localization.

In [ ]:

def temporal_iou(start_a, end_a, start_b, end_b):
    """
    Compute temporal Intersection over Union between two time intervals.

    IoU = intersection length / union length
    """

    intersection_start = max(start_a, start_b)
    intersection_end = min(end_a, end_b)

    intersection = max(0.0, intersection_end - intersection_start)

    union_start = min(start_a, start_b)
    union_end = max(end_a, end_b)

    union = max(1e-8, union_end - union_start)

    return intersection / union

In [ ]:

MANUAL_ANNOTATIONS_PATH = TASK3_OUTPUT_DIR / "manual_annotations.csv"

if not MANUAL_ANNOTATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Manual annotations file not found: {MANUAL_ANNOTATIONS_PATH}"
    )

manual_annotations_df = pd.read_csv(MANUAL_ANNOTATIONS_PATH)
if PLOT:
    print(f"Loaded manual annotations from: {MANUAL_ANNOTATIONS_PATH}")
    display(manual_annotations_df.head())

Loaded manual annotations from: outputs_vlm_event_detection\manual_annotations.csv


,video_name,event_id,event_description,start,end,start_seconds,end_seconds,subjectivity
0,TVSUM_9,1,Man gets into enclosed two-wheeled vehicle,00:00,00:06,0,6,2
1,TVSUM_9,2,Man speaking to camera,00:14,00:20,14,20,1
2,TVSUM_9,3,Man gets out of enclosed two-wheeled vehicle,00:26,00:32,26,32,2
3,TVSUM_9,4,Man speaking to camera,00:33,00:42,33,42,-1
4,TVSUM_9,5,Functionality of the vehicle is explained,00:43,00:57,43,57,2


In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Sentence embedding model loaded.")


def evaluate_moment_retrieval_with_iou(predictions_df, manual_df):
    """
    For each predicted moment:
    1. Find the most semantically similar manual event from the same video.
    2. Compute temporal IoU between the predicted segment and manual segment.
    """

    rows = []

    for video_name in predictions_df["video_name"].unique():
        pred_video_df = predictions_df[
            predictions_df["video_name"] == video_name
        ].copy()

        manual_video_df = manual_df[
            manual_df["video_name"] == video_name
        ].copy()

        if pred_video_df.empty or manual_video_df.empty:
            continue

        pred_queries = pred_video_df["base_query"].astype(str).tolist()
        manual_events = manual_video_df["event_description"].astype(str).tolist()

        pred_embeddings = embedding_model.encode(pred_queries)
        manual_embeddings = embedding_model.encode(manual_events)

        similarity_matrix = cosine_similarity(pred_embeddings, manual_embeddings)

        for i, (_, pred_row) in enumerate(pred_video_df.iterrows()):
            best_manual_idx = int(np.argmax(similarity_matrix[i]))
            best_similarity = float(similarity_matrix[i][best_manual_idx])
            manual_row = manual_video_df.iloc[best_manual_idx]

            iou = temporal_iou(
                pred_row["pred_start_seconds"],
                pred_row["pred_end_seconds"],
                manual_row["start_seconds"],
                manual_row["end_seconds"]
            )

            rows.append({
                "model_name": pred_row["model_name"],
                "prompt_name": pred_row["prompt_name"],
                "video_name": video_name,

                "vlm_event_id": pred_row["vlm_event_id"],
                "retrieval_query": pred_row["base_query"],
                "query_used": pred_row["query_used"],

                "pred_start_seconds": pred_row["pred_start_seconds"],
                "pred_end_seconds": pred_row["pred_end_seconds"],
                "pred_start": pred_row["pred_start"],
                "pred_end": pred_row["pred_end"],
                "confidence": pred_row["confidence"],

                "matched_manual_event_id": manual_row["event_id"],
                "matched_manual_event": manual_row["event_description"],
                "manual_start_seconds": manual_row["start_seconds"],
                "manual_end_seconds": manual_row["end_seconds"],
                "manual_start": manual_row["start"],
                "manual_end": manual_row["end"],
                "manual_subjectivity": manual_row["subjectivity"],

                "semantic_similarity": best_similarity,
                "temporal_iou": iou
            })

    return pd.DataFrame(rows)

Sentence embedding model loaded.


In [ ]:
if all_moment_predictions_df.empty:
    raise ValueError("No moment retrieval predictions found. Run Task 4 first.")

moment_iou_eval_df = evaluate_moment_retrieval_with_iou(
    predictions_df=all_moment_predictions_df,
    manual_df=manual_annotations_df
)


moment_iou_eval_df.to_csv(
    MOMENT_OUTPUT_DIR / "moment_retrieval_iou_evaluation.csv",
    index=False
)
if PLOT:
    display(moment_iou_eval_df)
    print("Saved IoU evaluation to:")
    print(MOMENT_OUTPUT_DIR / "moment_retrieval_iou_evaluation.csv")

,model_name,prompt_name,video_name,vlm_event_id,retrieval_query,query_used,pred_start_seconds,pred_end_seconds,pred_start,pred_end,confidence,matched_manual_event_id,matched_manual_event,manual_start_seconds,manual_end_seconds,manual_start,manual_end,manual_subjectivity,semantic_similarity,temporal_iou
0,CG-DETR,few_shot,TVSUM_9,1,A person sits next to a car,A person sits next to a car,1.5403,20.9329,00:02,00:21,0.9989,5,Functionality of the vehicle is explained,43,57,00:43,00:57,2,0.372733,0.000000
1,CG-DETR,few_shot,TVSUM_9,2,The person looks around,the moment when The person looks around,72.5142,89.8179,01:13,01:30,0.9996,2,Man speaking to camera,14,20,00:14,00:20,1,0.261481,0.000000
2,CG-DETR,few_shot,TVSUM_9,3,The car drives on the road,the moment when The car drives on the road,213.7529,228.6061,03:34,03:49,0.9882,13,Vehicle drives around bus depot,170,176,02:50,02:56,1,0.474448,0.000000
3,CG-DETR,few_shot,TVSUM_9,4,The car's interior is shown,a scene where The car's interior is shown,69.9871,90.2572,01:10,01:30,0.9947,5,Functionality of the vehicle is explained,43,57,00:43,00:57,2,0.435083,0.000000
4,CG-DETR,few_shot,TVSUM_9,5,The person looks at the camera,someone is involved in: The person looks at th...,71.9306,90.5704,01:12,01:31,0.9993,2,Man speaking to camera,14,20,00:14,00:20,1,0.569412,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,Moment-DETR,few_shot,TVSUM_12,13,The person applies hair gel to their hair again,The person applies hair gel to their hair again,395.1354,408.9828,06:35,06:49,0.9990,7,A pair of hair clippers being shown,394,400,06:34,06:40,2,0.428345,0.324679
132,Moment-DETR,few_shot,TVSUM_12,14,The person applies hair spray to their hair again,a scene where The person applies hair spray to...,102.4250,120.0150,01:42,02:00,0.9978,6,Hair cutting materials being shown with a pile...,341,347,05:41,05:47,2,0.387013,0.000000
133,Moment-DETR,few_shot,TVSUM_12,15,The person applies hair gel to their hair again,someone is involved in: The person applies hai...,105.7520,119.9699,01:46,02:00,0.9980,7,A pair of hair clippers being shown,394,400,06:34,06:40,2,0.428345,0.000000
134,Moment-DETR,few_shot,TVSUM_12,16,The person applies hair spray to their hair again,the moment when The person applies hair spray ...,119.8766,140.7893,02:00,02:21,0.9989,6,Hair cutting materials being shown with a pile...,341,347,05:41,05:47,2,0.387013,0.000000


Saved IoU evaluation to:
outputs_moment_retrieval\moment_retrieval_iou_evaluation.csv


In [ ]:
moment_iou_summary_df = (
    moment_iou_eval_df
    .groupby("model_name")
    .agg(
        num_predictions=("temporal_iou", "count"),
        mean_iou=("temporal_iou", "mean"),
        median_iou=("temporal_iou", "median"),
        max_iou=("temporal_iou", "max"),
        min_iou=("temporal_iou", "min"),
        mean_confidence=("confidence", "mean"),
        mean_semantic_similarity=("semantic_similarity", "mean")
    )
    .reset_index()
)

display(moment_iou_summary_df)

moment_iou_summary_df.to_csv(
    MOMENT_OUTPUT_DIR / "moment_retrieval_iou_summary_by_model.csv",
    index=False
)

,model_name,num_predictions,mean_iou,median_iou,max_iou,min_iou,mean_confidence,mean_semantic_similarity
0,CG-DETR,68,0.035487,0.0,0.902834,0.0,0.993734,0.456732
1,Moment-DETR,68,0.027919,0.0,0.818843,0.0,0.997135,0.456732


In [ ]:

moment_iou_summary_by_video_df = (
    moment_iou_eval_df
    .groupby(["model_name", "video_name"])
    .agg(
        num_predictions=("temporal_iou", "count"),
        mean_iou=("temporal_iou", "mean"),
        median_iou=("temporal_iou", "median"),
        mean_confidence=("confidence", "mean"),
        mean_semantic_similarity=("semantic_similarity", "mean")
    )
    .reset_index()
)

display(moment_iou_summary_by_video_df)

moment_iou_summary_by_video_df.to_csv(
    MOMENT_OUTPUT_DIR / "moment_retrieval_iou_summary_by_model_and_video.csv",
    index=False
)

,model_name,video_name,num_predictions,mean_iou,median_iou,mean_confidence,mean_semantic_similarity
0,CG-DETR,TVSUM_10,16,0.000000,0.0,0.995144,0.445252
1,CG-DETR,TVSUM_11,17,0.005083,0.0,0.988629,0.533032
2,CG-DETR,TVSUM_12,17,0.030651,0.0,0.998876,0.347217
3,CG-DETR,TVSUM_9,18,0.100315,0.0,0.992444,0.498307
4,Moment-DETR,TVSUM_10,16,0.010430,0.0,0.997856,0.445252
5,Moment-DETR,TVSUM_11,17,0.001326,0.0,0.996759,0.533032
6,Moment-DETR,TVSUM_12,17,0.029808,0.0,0.998824,0.347217
7,Moment-DETR,TVSUM_9,18,0.066795,0.0,0.995256,0.498307


In [ ]:
IOU_THRESHOLD = 0.3

moment_failure_analysis_df = moment_iou_eval_df[
    moment_iou_eval_df["temporal_iou"] < IOU_THRESHOLD
].copy()

moment_failure_analysis_df = moment_failure_analysis_df.sort_values(
    by=["model_name", "video_name", "manual_subjectivity", "temporal_iou"],
    ascending=[True, True, False, True]
)


moment_failure_analysis_df.to_csv(
    MOMENT_OUTPUT_DIR / "moment_retrieval_failure_analysis.csv",
    index=False
)

if PLOT:
    display(moment_failure_analysis_df)
    print(f"Number of low-IoU predictions: {len(moment_failure_analysis_df)}")

,model_name,prompt_name,video_name,vlm_event_id,retrieval_query,query_used,pred_start_seconds,pred_end_seconds,pred_start,pred_end,confidence,matched_manual_event_id,matched_manual_event,manual_start_seconds,manual_end_seconds,manual_start,manual_end,manual_subjectivity,semantic_similarity,temporal_iou
36,CG-DETR,few_shot,TVSUM_10,1,An electric car is shown being charged,a video of An electric car is shown being charged,6.4564,28.3717,00:06,00:28,0.9992,3,Cable being plugged into a car,32,33,00:32,00:33,2,0.528524,0.000000
45,CG-DETR,few_shot,TVSUM_10,10,A man speaks about the future of electric cars,a video of A man speaks about the future of el...,125.1200,130.0643,02:05,02:10,0.9918,6,Man talking with cars in the background,88,102,01:28,01:42,2,0.532167,0.000000
43,CG-DETR,few_shot,TVSUM_10,8,Attendees are shown taking notes during a meeting,someone is involved in: Attendees are shown ta...,120.0000,122.9711,02:00,02:03,0.9974,1,Woman talking on a set,5,25,00:05,00:25,1,0.286537,0.000000
44,CG-DETR,few_shot,TVSUM_10,9,Attendees are shown interacting with a display...,Attendees are shown interacting with a display...,130.8149,133.8710,02:11,02:14,0.9993,1,Woman talking on a set,5,25,00:05,00:25,1,0.270632,0.000000
46,CG-DETR,few_shot,TVSUM_10,11,A woman demonstrates how to use a charging sta...,A woman demonstrates how to use a charging sta...,63.9611,75.9530,01:04,01:16,0.9985,1,Woman talking on a set,5,25,00:05,00:25,1,0.298770,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24,Moment-DETR,few_shot,TVSUM_9,7,The car's interior is shown again,The car's interior is shown again,165.4360,183.4887,02:45,03:03,0.9979,10,Smart display shows vehicle data,119,123,01:59,02:03,1,0.354037,0.000000
25,Moment-DETR,few_shot,TVSUM_9,8,The car drives on the road once more,The car drives on the road once more,71.9606,89.8794,01:12,01:30,0.9982,13,Vehicle drives around bus depot,170,176,02:50,02:56,1,0.451756,0.000000
26,Moment-DETR,few_shot,TVSUM_9,9,A woman drives the car,a video of A woman drives the car,0.0636,42.8368,00:00,00:43,0.9951,15,Woman exits the vehicle in sunny weather,210,213,03:30,03:33,1,0.438991,0.000000
22,Moment-DETR,few_shot,TVSUM_9,5,The person looks at the camera,a video of The person looks at the camera,8.2833,42.2595,00:08,00:42,0.9787,2,Man speaking to camera,14,20,00:14,00:20,1,0.569412,0.176594


Number of low-IoU predictions: 131


In [ ]:
failure_summary_by_model_df = (
    moment_failure_analysis_df
    .groupby("model_name")
    .agg(
        failed_predictions=("temporal_iou", "count"),
        mean_failed_iou=("temporal_iou", "mean"),
        mean_failed_semantic_similarity=("semantic_similarity", "mean"),
        mean_failed_confidence=("confidence", "mean")
    )
    .reset_index()
)

display(failure_summary_by_model_df)

failure_summary_by_model_df.to_csv(
    MOMENT_OUTPUT_DIR / "moment_retrieval_failure_summary_by_model.csv",
    index=False
)

,model_name,failed_predictions,mean_failed_iou,mean_failed_semantic_similarity,mean_failed_confidence
0,CG-DETR,65,0.001329,0.462985,0.993906
1,Moment-DETR,66,0.011439,0.457714,0.997117


In [ ]:
worst_failures_df = moment_iou_eval_df.sort_values(
    by="temporal_iou",
    ascending=True
).head(20)
if PLOT:
    display(worst_failures_df[[
        "model_name",
        "video_name",
        "retrieval_query",
        "matched_manual_event",
        "pred_start",
        "pred_end",
        "manual_start",
        "manual_end",
        "semantic_similarity",
        "temporal_iou",
        "confidence"
    ]])

worst_failures_df.to_csv(
    MOMENT_OUTPUT_DIR / "moment_retrieval_worst_failures.csv",
    index=False
)

,model_name,video_name,retrieval_query,matched_manual_event,pred_start,pred_end,manual_start,manual_end,semantic_similarity,temporal_iou,confidence
0,CG-DETR,TVSUM_9,A person sits next to a car,Functionality of the vehicle is explained,00:02,00:21,00:43,00:57,0.372733,0.0,0.9989
99,Moment-DETR,TVSUM_11,The person prepares the fifth dog for grooming,A dog getting brushed,02:22,02:30,01:37,01:38,0.489168,0.0,0.9959
98,Moment-DETR,TVSUM_11,The person interacts with a fifth dog,A woman talking while petting a dog,00:39,01:06,00:05,00:09,0.502452,0.0,0.9984
97,Moment-DETR,TVSUM_11,The person grooms the fourth dog,A woman trimming a dog's paws,00:40,01:13,01:33,01:34,0.523128,0.0,0.9958
96,Moment-DETR,TVSUM_11,The person prepares the fourth dog for grooming,A woman trimming a dog's paws,02:08,02:20,01:33,01:34,0.513103,0.0,0.9969
95,Moment-DETR,TVSUM_11,The person interacts with a fourth dog,A woman talking while petting a dog,02:19,02:34,00:05,00:09,0.510717,0.0,0.9960
94,Moment-DETR,TVSUM_11,The person grooms the third dog,A woman trimming a dog's paws,02:28,02:34,01:33,01:34,0.521828,0.0,0.9991
92,Moment-DETR,TVSUM_11,The person interacts with a third dog,A woman talking while petting a dog,02:22,02:35,00:05,00:09,0.547346,0.0,0.9989
91,Moment-DETR,TVSUM_11,The person grooms the second dog,A woman trimming a dog's paws,02:05,02:12,01:33,01:34,0.513103,0.0,0.9970
90,Moment-DETR,TVSUM_11,The person prepares the second dog for grooming,A woman trimming a dog's paws,02:15,02:29,01:33,01:34,0.497312,0.0,0.9989


In [ ]:
import cv_utils
from moviepy import VideoFileClip

if not hasattr(VideoFileClip, "subclip") and hasattr(VideoFileClip, "subclipped"):
    VideoFileClip.subclip = VideoFileClip.subclipped

cv_utils.VideoFileClip = VideoFileClip

print("Patch applied.")
print("cv_utils VideoFileClip has subclip:", hasattr(cv_utils.VideoFileClip, "subclip"))

In [ ]:
def clamp_timestamps_to_video(video_path, timestamps):
    """
    Make sure predicted timestamps stay inside the actual video duration.
    """
    duration = get_video_duration_seconds(video_path)

    cleaned = []

    for start, end in timestamps:
        start = float(start)
        end = float(end)

        start = max(0, min(start, duration))
        end = max(0, min(end, duration))

        if end > start:
            cleaned.append((start, end))

    return cleaned

In [ ]:
SUMMARY_OUTPUT_DIR = output_dir("outputs_video_summary")

SELECTED_SUMMARY_MODEL = "CG-DETR"
MAX_SEGMENTS_PER_VIDEO = 6
MIN_SEGMENT_SECONDS = 1
MAX_SEGMENT_SECONDS = 60

summary_rows = []

for video_name, video_path in VIDEOS.items():
    print("\n" + "=" * 80)
    print(f"Creating summary for {video_name}")
    print("=" * 80)

    df = all_moment_predictions_df[
        (all_moment_predictions_df["video_name"] == video_name) &
        (all_moment_predictions_df["model_name"] == SELECTED_SUMMARY_MODEL)
    ].copy()

    if df.empty:
        print(f"No predictions found for {video_name}")
        continue

    df["duration"] = df["pred_end_seconds"] - df["pred_start_seconds"]

    # Filter invalid / unreasonable segments
    df = df[
        (df["duration"] >= MIN_SEGMENT_SECONDS) &
        (df["duration"] <= MAX_SEGMENT_SECONDS)
    ].copy()

    if df.empty:
        print(f"No valid segments found for {video_name}")
        continue

    # Select most confident segments
    df = df.sort_values("confidence", ascending=False).head(MAX_SEGMENTS_PER_VIDEO)

    # Sort by original time for coherence
    df = df.sort_values("pred_start_seconds")

    timestamps = list(zip(df["pred_start_seconds"], df["pred_end_seconds"]))

    # Important: clamp predicted timestamps to actual video duration
    timestamps = clamp_timestamps_to_video(video_path, timestamps)

    if len(timestamps) == 0:
        print(f"No valid timestamps after clamping for {video_name}")
        continue

    display(df[[
        "video_name",
        "model_name",
        "base_query",
        "pred_start",
        "pred_end",
        "confidence",
        "duration"
    ]])

    output_path = SUMMARY_OUTPUT_DIR / f"{video_name}_{SELECTED_SUMMARY_MODEL}_summary.mp4"

    stitch_video_segments(
        video_path=video_path,
        timestamps=timestamps,
        output_path=output_path
    )

    for _, row in df.iterrows():
        summary_rows.append({
            "video_name": video_name,
            "model_name": SELECTED_SUMMARY_MODEL,
            "retrieval_query": row["base_query"],
            "summary_start": row["pred_start"],
            "summary_end": row["pred_end"],
            "summary_start_seconds": row["pred_start_seconds"],
            "summary_end_seconds": row["pred_end_seconds"],
            "duration": row["duration"],
            "confidence": row["confidence"],
            "summary_video_path": str(output_path)
        })

summary_segments_df = pd.DataFrame(summary_rows)

display(summary_segments_df)

summary_segments_df.to_csv(
    SUMMARY_OUTPUT_DIR / "summary_segments_used.csv",
    index=False
)

In [ ]:
# Task 6: Summary Metrics for a/b/c/d

summary_metrics_rows = []

for video_name in summary_segments_df["video_name"].unique():
    summary_video_df = summary_segments_df[
        summary_segments_df["video_name"] == video_name
    ].copy()

    manual_video_df = manual_annotations_df[
        manual_annotations_df["video_name"] == video_name
    ].copy()

    num_selected_segments = len(summary_video_df)
    num_manual_events = len(manual_video_df)

    total_summary_duration = summary_video_df["duration"].sum()
    average_segment_duration = summary_video_df["duration"].mean()

    # a) Relevance proxy:
    # average confidence of selected retrieved clips
    relevance_score = summary_video_df["confidence"].mean()

    # b) Coverage proxy:
    # how many selected clips compared to number of manual events
    coverage_ratio = (
        num_selected_segments / num_manual_events
        if num_manual_events > 0 else 0
    )

    # c) Conciseness proxy:
    # lower total duration means more concise
    conciseness_ratio = (
        average_segment_duration / total_summary_duration
        if total_summary_duration > 0 else 0
    )

    # d) Coherence proxy:
    # whether clips are sorted chronologically
    coherence_order = summary_video_df[
        "summary_start_seconds"
    ].is_monotonic_increasing

    summary_metrics_rows.append({
        "video_name": video_name,

        "a_relevance_score_mean_confidence": relevance_score,
        "b_coverage_ratio_selected_vs_manual": coverage_ratio,
        "c_total_summary_duration_seconds": total_summary_duration,
        "c_average_clip_duration_seconds": average_segment_duration,
        "d_coherence_chronological_order": coherence_order,
        "selected_segments": num_selected_segments,
        "manual_events": num_manual_events
    })

summary_metrics_df = pd.DataFrame(summary_metrics_rows)

display(summary_metrics_df)

summary_metrics_df.to_csv(
    SUMMARY_OUTPUT_DIR / "summary_metrics_abcd.csv",
    index=False
)